In [1]:
# ============================
# Quarterly Simulation (40 visits = 10 years)
# ============================
import os
import numpy as np
import pandas as pd

# -------- Global setup (QUARTERLY) --------
VISIT_LEN_YEARS = 0.25                  # 1 visit = quarter
N_VISITS = 40                           # 10 years → 40 quarters (1..40)
QOL_VISITS = set(range(4, 41, 4))       # annual QOL at quarters: 4,8,12,...,40
VISIT_TO_YEARS = {v: f"{v//4} year" + ("s" if v//4 > 1 else "") for v in sorted(QOL_VISITS)}

# -------- Tunable parameters (REBALANCED) --------
AGE_PENALTY_PER_YEAR = 0.015            # QOL penalty per year >75
NOISE_SIGMA = 0.03                      # random QOL noise

# Dose burden on QOL (independent of efficacy)
DOSE_PEN_FULL    = (0.040, 0.060)       # (fit, frail) penalty when on full dose  (↑)
DOSE_PEN_REDUCED = 0.006                # small reduced-burden

# Efficacy boost only when disease activity (clinical_test=1)
EFFICACY_FULL    = 0.015                # was 0.030 (↓)
EFFICACY_REDUCED = 0.008                # was 0.010 (slightly ↓)

# Toxicity → immediate drop + partial, slower recovery (carry-over)
SEVERE_TOX_QOL_HIT_FIT   = 0.05         # immediate QOL hit if fit
SEVERE_TOX_QOL_HIT_FRAIL = 0.10         # immediate QOL hit if frail (↑)
CARRYOVER_DAMAGE_FRACTION = 0.80        # fraction of hit that persists as "scar" (↑)
QOL_DAMAGE_RECOVERY_PER_QUARTER = 0.20  # heals 20% of stored damage per quarter (slower) (↓)

# Test inertia (smooths test flips)
TEST_INERTIA_RHO = 0.60   # was 0.50 (more inertia → fewer flips)

# Dose-change inertia (semi-Markov rules)
DOSE_MIN_DWELL = 1        # must stay ≥ this many quarters before change (↑)
ESCALATE_POS_STREAK = 2   # need ≥3 consecutive positives to escalate (0→1) (↑)
DEESCALATE_NEG_STREAK = 1 # ≥2 consecutive negatives to de-escalate (1→0)

# === cumulative full-dose exposure (chronic toxicity, γραμμικό) ===
# cum_dose_cat είναι συνεχές (float) ≈ "πόσες full-dose επισκέψεις έχει φάει"
CUM_DOSE_SLOPE = 0.005                # +0.5% στο P(test=1) ανά full-dose visit
MAX_CUM_DOSE_EFFECT = 0.25            # cap στο +25% επιπλέον πιθανότητα
CUM_DOSE_RECOVERY_PER_QUARTER = 0.10  # ανά quarter σε reduced/0 μειώνεται λίγο

def update_cum_dose_cat(cum_dose_cat, dose):
    """
    cum_dose_cat: συνεχές μέγεθος (float) που μετρά cumulative full-dose exposure.

    - Αν dose == 1 (full): προσθέτουμε 1.0 μονάδα (ένα ακόμα full quarter).
    - Αν dose == 0 (reduced/stop): αφαιρούμε CUM_DOSE_RECOVERY_PER_QUARTER (μέχρι min 0).
    """
    if dose == 1:
        cum_dose_cat = cum_dose_cat + 1.0
    else:
        cum_dose_cat = max(0.0, cum_dose_cat - CUM_DOSE_RECOVERY_PER_QUARTER)
    return float(cum_dose_cat)

def cum_dose_effect_on_p_pos(cum_dose_cat):
    """
    Γραμμική επίδραση της cumulative dose στην πιθανότητα bad test (test=1).

    extra = CUM_DOSE_SLOPE * cum_dose_cat, capped στο MAX_CUM_DOSE_EFFECT.
    Π.χ. με CUM_DOSE_SLOPE=0.005:
        10 full visits → +0.05
        20 full visits → +0.10
        50 full visits → +0.25 (cap)
    """
    extra = CUM_DOSE_SLOPE * cum_dose_cat
    return float(np.clip(extra, 0.0, MAX_CUM_DOSE_EFFECT))

# --- Toxicity probabilities per QUARTER (relative risk reduced on reduced dose)
def p_toxicity_g34(age, dose):
    base = 0.12 if age < 65 else 0.17   # per quarter (proxy; ↑ vs before)
    rr   = 1.0 if dose == 1 else 0.55   # reduced = 45% risk reduction (more protective)
    return float(np.clip(base * rr, 0, 1))

def p_toxicity_death(age, dose):
    base = 0.0015 if age < 65 else 0.0145  # per quarter Grade 5 (kept base)
    rr   = 1.0 if dose == 1 else 0.50      # reduced halves the risk (↑ effect)
    return float(np.clip(base * rr, 0, 1))

# -------- Clinical test evolution (disease control) --------
def base_next_test_positive_prob(current_test, dose, cum_dose_cat=0.0):
    """
    Softer dose effects vs earlier version + cumulative toxicity effect.
    """
    if current_test == 1:
        p_pos = 0.40
        p_pos *= 0.92 if dose == 1 else 1.03  # full slightly better; reduced slightly worse
    else:
        p_pos = 0.20
        p_pos *= 0.95 if dose == 1 else 1.05

    # chronic toxicity effect (μικρό, γραμμικό penalty ανά full visit)
    p_pos += cum_dose_effect_on_p_pos(cum_dose_cat)

    return float(np.clip(p_pos, 0.05, 0.99))

def next_test_positive_prob_with_inertia(current_test, dose, cum_dose_cat, rho=TEST_INERTIA_RHO):
    """
    P(next=1) = rho*current_test + (1-rho)*base_prob(cum_dose_cat)
    """
    base_p = base_next_test_positive_prob(current_test, dose, cum_dose_cat)
    return float(np.clip(rho*current_test + (1.0 - rho)*base_p, 0.01, 0.99))

# -------- QOL generator (with efficacy & carry-over damage) --------
def simulate_qol(frailty, age, dose, clinical_test, rng, carryover_damage):
    """
    QOL ∈ [0,1], quarterly.
    - Fit: U(0.55, 0.90) | Frail: U(0.35, 0.70)
    - Age penalty from 75+: AGE_PENALTY_PER_YEAR * (age-75)
    - Dose penalty (full>reduced) + efficacy when test=1 + noise
    - Subtract stored carry-over 'scar' damage
    """
    lo, hi = (0.55, 0.90) if frailty == 0 else (0.35, 0.70)
    base_qol = rng.uniform(lo, hi)
    age_pen  = max(0.0, (age - 75) * AGE_PENALTY_PER_YEAR)

    # dose burden by frailty/dose
    dose_pen = (
        DOSE_PEN_FULL[0] if (dose == 1 and frailty == 0) else
        DOSE_PEN_FULL[1] if (dose == 1 and frailty == 1) else
        DOSE_PEN_REDUCED
    )

    # efficacy only if current test is positive
    efficacy = (EFFICACY_FULL if dose == 1 else EFFICACY_REDUCED) if clinical_test == 1 else 0.0

    noise = rng.normal(0, NOISE_SIGMA)
    qol = base_qol - age_pen - dose_pen + efficacy + noise - carryover_damage
    return float(np.clip(qol, 0, 1))

# -------- Frailty dynamics (less dose-driven) --------
def frailty_transition(frail, age, dose, clinical_test, pos_streak, rng):
    """
    Slower transitions with modest dose effects.
    """
    if frail == 0:
        p = 0.014 if age <= 75 else (0.022 if age <= 80 else 0.036)
        if dose == 1: p -= 0.003               # smaller protection from full
        if pos_streak >= 2:                     # sustained activity raises risk
            p += 0.008 if dose == 0 else 0.003
        return 1 if rng.random() < np.clip(p, 0, 1) else 0
    else:
        r = 0.010 if age <= 75 else (0.008 if age <= 80 else 0.006)
        if dose == 1: r += 0.004               # smaller recovery boost on full
        if clinical_test == 0: r += 0.003      # good tests help recovery
        return 0 if rng.random() < np.clip(r, 0, 1) else 1

# -------- Initial dose policy --------
def assign_initial_dose(frail, rng):
    return 1 if frail == 0 else int(rng.random() < 0.5)  # fit→full, frail→50/50

# -------- Main simulator --------
def simulate_data(n_patients=500, n_visits=N_VISITS, seed=42, save_csv=True, csv_path='Fixed_patient_data.csv'):
    rng = np.random.default_rng(seed)
    ids = np.arange(1, n_patients + 1)
    ages = rng.integers(71, 96, size=n_patients)       # starting ages
    frail0 = rng.choice([1, 0], size=n_patients, p=[0.7, 0.3])
    dose0 = np.array([assign_initial_dose(f, rng) for f in frail0])
    test0 = rng.integers(0, 2, size=n_patients)        # initial clinical test (0/1)

    print(f"Baseline: {np.mean(frail0==0)*100:.1f}% fit / {np.mean(frail0==1)*100:.1f}% frail (n={n_patients})")

    records = []
    for pid, age0, frail_init, dose_init, test_init in zip(ids, ages, frail0, dose0, test0):
        age   = float(age0)
        frail = int(frail_init)
        dose  = int(dose_init)
        test  = int(test_init)
        alive = True
        cum_qaly = 0.0

        # inertia trackers
        dwell_dose = 0
        pos_streak = 1 if test == 1 else 0
        neg_streak = 1 if test == 0 else 0

        # persistent toxicity damage that heals gradually
        carryover_damage = 0.0

        # cumulative full-dose exposure counter
        cum_dose_cat = 0.0

        for visit_idx in range(n_visits):  # 0..39
            visit = visit_idx + 1          # 1..40
            if not alive:
                break

            # heal part of the stored damage each quarter
            carryover_damage *= (1.0 - QOL_DAMAGE_RECOVERY_PER_QUARTER)

            # QOL before mortality check (includes carry-over damage)
            qol = simulate_qol(frail, age, dose, test, rng, carryover_damage)

            # Toxicity events (per quarter)
            died_tox = rng.random() < p_toxicity_death(age, dose)
            severe_tox = False
            if not died_tox:
                severe_tox = rng.random() < p_toxicity_g34(age, dose)
                if severe_tox:
                    hit = SEVERE_TOX_QOL_HIT_FRAIL if frail == 1 else SEVERE_TOX_QOL_HIT_FIT
                    # immediate drop + store a fraction as "scar"
                    qol = max(0.0, qol - hit)
                    carryover_damage += hit * CARRYOVER_DAMAGE_FRACTION

            # QALYs accrue per quarter
            qaly = qol * VISIT_LEN_YEARS
            cum_qaly += qaly

            records.append({
                'patient_id': pid,
                'age': round(age, 2),
                'visit': visit,                         # 1..40
                'frailty': frail,
                'clinical_test': int(test),
                'full_dose': int(dose),
                'cum_dose_cat': float(cum_dose_cat),    # NEW: cumulative exposure state
                'QOL': qol if visit in QOL_VISITS else np.nan,
                'qaly_quarter': qaly,
                'cum_qaly': cum_qaly,
                'severe_tox': int(severe_tox),
                'death_tox': int(died_tox)
            })

            if died_tox:
                alive = False
                break

            # ---- evolve state/time for next quarter ----

            # 0) update cumulative dose *μετά* από αυτό το quarter (ανάλογα με το dose)
            cum_dose_cat = update_cum_dose_cat(cum_dose_cat, dose)

            # 1) clinical test with inertia + cumulative toxicity effect
            p_next_pos = next_test_positive_prob_with_inertia(test, dose, cum_dose_cat, rho=TEST_INERTIA_RHO)
            test = 1 if rng.random() < p_next_pos else 0
            if test == 1:
                pos_streak += 1; neg_streak = 0
            else:
                neg_streak += 1; pos_streak = 0

            # 2) frailty transition (slow + depends on activity streak)
            frail = frailty_transition(frail, age, dose, test, pos_streak, rng)

            # 3) Dose-change rules with dwell + streaks
            if dwell_dose >= DOSE_MIN_DWELL:
                # Escalate if ≥3 consecutive positives and currently reduced
                if dose == 0 and pos_streak >= ESCALATE_POS_STREAK:
                    dose = 1
                    dwell_dose = 0
                # De-escalate if ≥2 consecutive negatives and currently full
                elif dose == 1 and neg_streak >= DEESCALATE_NEG_STREAK:
                    dose = 0
                    dwell_dose = 0
                else:
                    dwell_dose += 1
            else:
                dwell_dose += 1

            # 4) age increases by one quarter
            age += VISIT_LEN_YEARS

    df = pd.DataFrame(records)
    if save_csv:
        if os.path.exists(csv_path):
            os.remove(csv_path)
        df.to_csv(csv_path, index=False)
        print(f"Saved '{csv_path}' ({len(df)} rows).")
    return df

# -------- Run once and basic sanity checks --------
if __name__ == "__main__":
    df = simulate_data(n_patients=500, n_visits=N_VISITS, seed=42, save_csv=True)

    print("\n--- Sanity checks ---")
    print("Columns:", df.columns.tolist())
    print("Patients:", df['patient_id'].nunique(), "| Max visit:", int(df['visit'].max()))
    print("Severe tox events:", int(df['severe_tox'].sum()))
    print("Deaths from toxicity:", int(df['death_tox'].sum()))
    print("Patients with any death:", int(df.groupby('patient_id')['death_tox'].max().sum()))
    print("Last-visit distribution (top):")
    print(df.groupby('patient_id')['visit'].max().value_counts().head())

    # Quick mapping preview (first 5 QOL rows with year labels)
    preview = (
        df[df['visit'].isin(QOL_VISITS)]
          .assign(year=lambda d: d['visit'].map(VISIT_TO_YEARS))
    )[["patient_id","visit","year","QOL"]].head()
    print("\nPreview QOL rows:\n", preview)


In [2]:
# ============================
# Quarterly Simulation (40 visits = 10 years)
# ============================

import os
import numpy as np
import pandas as pd

# -------- Global setup (QUARTERLY) --------
VISIT_LEN_YEARS = 0.25          # 1 visit = quarter
N_VISITS = 40                   # 10 years → 40 quarters (1..40)
QOL_VISITS = set(range(4, 41, 4))  # annual QOL at quarters: 4,8,12,...,40
VISIT_TO_YEARS = {
    v: f"{v // 4} year" + ("s" if v // 4 > 1 else "")
    for v in sorted(QOL_VISITS)
}

# -------- Tunable parameters (REBALANCED) --------
AGE_PENALTY_PER_YEAR = 0.015    # QOL penalty per year >75
NOISE_SIGMA = 0.03              # random QOL noise

# Dose burden on QOL (independent of efficacy)
DOSE_PEN_FULL = (0.040, 0.060)  # (fit, frail) penalty when on full dose  (↑)
DOSE_PEN_REDUCED = 0.006        # small reduced-burden

# Efficacy boost only when disease activity (clinical_test=1)
EFFICACY_FULL = 0.015           # was 0.030 (↓)
EFFICACY_REDUCED = 0.008        # was 0.010 (slightly ↓)

# Toxicity → immediate drop + partial, slower recovery (carry-over)
SEVERE_TOX_QOL_HIT_FIT = 0.15        # immediate QOL hit if fit
SEVERE_TOX_QOL_HIT_FRAIL = 0.22      # immediate QOL hit if frail (↑)
CARRYOVER_DAMAGE_FRACTION = 0.80     # fraction of hit that persists as "scar" (↑)
QOL_DAMAGE_RECOVERY_PER_QUARTER = 0.20  # heals 20% of stored damage per quarter (slower) (↓)

# Test inertia (smooths test flips)
TEST_INERTIA_RHO = 0.60         # was 0.50 (more inertia → fewer flips)

# Dose-change inertia (semi-Markov rules)
DOSE_MIN_DWELL = 2              # must stay ≥ this many quarters before change (↑)
ESCALATE_POS_STREAK = 3         # need ≥3 consecutive positives to escalate (0→1) (↑)
DEESCALATE_NEG_STREAK = 2       # ≥2 consecutive negatives to de-escalate (1→0)


# --- Toxicity probabilities per QUARTER (relative risk reduced on reduced dose)
def p_toxicity_g34(age, dose):
    base = 0.12 if age < 65 else 0.17   # per quarter (proxy; ↑ vs before)
    rr = 1.0 if dose == 1 else 0.55     # reduced = 45% risk reduction (more protective)
    return float(np.clip(base * rr, 0, 1))


def p_toxicity_death(age, dose):
    base = 0.0015 if age < 65 else 0.0145  # per quarter Grade 5 (kept base)
    rr = 1.0 if dose == 1 else 0.50        # reduced halves the risk (↑ effect)
    return float(np.clip(base * rr, 0, 1))


# -------- Clinical test evolution (disease control) --------
def base_next_test_positive_prob(current_test, dose):
    """
    Softer dose effects vs earlier version.
    """
    if current_test == 1:
        p_pos = 0.40
        # full slightly better; reduced slightly worse
        p_pos *= 0.92 if dose == 1 else 1.03
    else:
        p_pos = 0.20
        p_pos *= 0.95 if dose == 1 else 1.05
    return float(np.clip(p_pos, 0.05, 0.95))


def next_test_positive_prob_with_inertia(current_test, dose, rho=TEST_INERTIA_RHO):
    """
    P(next=1) = rho*current_test + (1-rho)*base_prob
    """
    base_p = base_next_test_positive_prob(current_test, dose)
    return float(np.clip(rho * current_test + (1.0 - rho) * base_p, 0.01, 0.99))


# -------- QOL generator (with efficacy & carry-over damage) --------
def simulate_qol(frailty, age, dose, clinical_test, rng, carryover_damage):
    """
    QOL ∈ [0,1], quarterly.
    - Fit: U(0.55, 0.90) | Frail: U(0.35, 0.70)
    - Age penalty from 75+: AGE_PENALTY_PER_YEAR * (age-75)
    - Dose penalty (full>reduced) + efficacy when test=1 + noise
    - Subtract stored carry-over 'scar' damage
    """
    lo, hi = (0.55, 0.90) if frailty == 0 else (0.35, 0.70)
    base_qol = rng.uniform(lo, hi)
    age_pen = max(0.0, (age - 75) * AGE_PENALTY_PER_YEAR)

    # dose burden by frailty/dose
    dose_pen = (
        DOSE_PEN_FULL[0] if (dose == 1 and frailty == 0) else
        DOSE_PEN_FULL[1] if (dose == 1 and frailty == 1) else
        DOSE_PEN_REDUCED
    )

    # efficacy only if current test is positive
    efficacy = (EFFICACY_FULL if dose == 1 else EFFICACY_REDUCED) if clinical_test == 1 else 0.0

    noise = rng.normal(0, NOISE_SIGMA)
    qol = base_qol - age_pen - dose_pen + efficacy + noise - carryover_damage
    return float(np.clip(qol, 0, 1))


# -------- Frailty dynamics (less dose-driven) --------
def frailty_transition(frail, age, dose, clinical_test, pos_streak, rng):
    """
    Slower transitions with modest dose effects.
    """
    if frail == 0:
        p = 0.014 if age <= 75 else (0.022 if age <= 80 else 0.036)
        if dose == 1:
            p -= 0.003  # smaller protection from full
        if pos_streak >= 2:  # sustained activity raises risk
            p += 0.008 if dose == 0 else 0.003
        return 1 if rng.random() < np.clip(p, 0, 1) else 0
    else:
        r = 0.010 if age <= 75 else (0.008 if age <= 80 else 0.006)
        if dose == 1:
            r += 0.004  # smaller recovery boost on full
        if clinical_test == 0:
            r += 0.003  # good tests help recovery
        return 0 if rng.random() < np.clip(r, 0, 1) else 1


# -------- Initial dose policy --------
def assign_initial_dose(frail, rng):
    # fit → full, frail → 50/50
    return 1 if frail == 0 else int(rng.random() < 0.5)


# -------- Main simulator --------
def simulate_data(
    n_patients=500,
    n_visits=N_VISITS,
    seed=42,
    save_csv=True,
    csv_path='Fixed_patient_data.csv'
):
    rng = np.random.default_rng(seed)
    ids = np.arange(1, n_patients + 1)
    ages = rng.integers(71, 96, size=n_patients)  # starting ages
    frail0 = rng.choice([1, 0], size=n_patients, p=[0.7, 0.3])
    dose0 = np.array([assign_initial_dose(f, rng) for f in frail0])
    test0 = rng.integers(0, 2, size=n_patients)  # initial clinical test (0/1)

    print(
        f"Baseline: {np.mean(frail0 == 0) * 100:.1f}% fit / "
        f"{np.mean(frail0 == 1) * 100:.1f}% frail (n={n_patients})"
    )

    records = []

    for pid, age0, frail_init, dose_init, test_init in zip(ids, ages, frail0, dose0, test0):
        age = float(age0)
        frail = int(frail_init)
        dose = int(dose_init)
        test = int(test_init)
        alive = True
        cum_qaly = 0.0

        # inertia trackers
        dwell_dose = 0
        pos_streak = 1 if test == 1 else 0
        neg_streak = 1 if test == 0 else 0

        # persistent toxicity damage that heals gradually
        carryover_damage = 0.0

        for visit_idx in range(n_visits):  # 0..39
            visit = visit_idx + 1          # 1..40
            if not alive:
                break

            # heal part of the stored damage each quarter
            carryover_damage *= (1.0 - QOL_DAMAGE_RECOVERY_PER_QUARTER)

            # QOL before mortality check (includes carry-over damage)
            qol = simulate_qol(frail, age, dose, test, rng, carryover_damage)

            # Toxicity events (per quarter)
            died_tox = rng.random() < p_toxicity_death(age, dose)
            severe_tox = False
            if not died_tox:
                severe_tox = rng.random() < p_toxicity_g34(age, dose)
                if severe_tox:
                    hit = SEVERE_TOX_QOL_HIT_FRAIL if frail == 1 else SEVERE_TOX_QOL_HIT_FIT
                    # immediate drop + store a fraction as "scar"
                    qol = max(0.0, qol - hit)
                    carryover_damage += hit * CARRYOVER_DAMAGE_FRACTION

            # QALYs accrue per quarter
            qaly = qol * VISIT_LEN_YEARS
            cum_qaly += qaly

            records.append({
                'patient_id': pid,
                'age': round(age, 2),
                'visit': visit,                      # 1..40
                'frailty': frail,
                'clinical_test': int(test),
                'full_dose': int(dose),
                'QOL': qol if visit in QOL_VISITS else np.nan,
                'qaly_quarter': qaly,
                'cum_qaly': cum_qaly,
                'severe_tox': int(severe_tox),
                'death_tox': int(died_tox)
            })

            if died_tox:
                alive = False
                break

            # ---- evolve state/time for next quarter ----

            # 1) clinical test with inertia (smoother)
            p_next_pos = next_test_positive_prob_with_inertia(
                test,
                dose,
                rho=TEST_INERTIA_RHO
            )
            test = 1 if rng.random() < p_next_pos else 0
            if test == 1:
                pos_streak += 1
                neg_streak = 0
            else:
                neg_streak += 1
                pos_streak = 0

            # 2) frailty transition (slow + depends on activity streak)
            frail = frailty_transition(frail, age, dose, test, pos_streak, rng)

            # 3) Dose-change rules with dwell + streaks
            if dwell_dose >= DOSE_MIN_DWELL:
                # Escalate if ≥3 consecutive positives and currently reduced
                if dose == 0 and pos_streak >= ESCALATE_POS_STREAK:
                    dose = 1
                    dwell_dose = 0
                # De-escalate if ≥2 consecutive negatives and currently full
                elif dose == 1 and neg_streak >= DEESCALATE_NEG_STREAK:
                    dose = 0
                    dwell_dose = 0
                else:
                    dwell_dose += 1
            else:
                dwell_dose += 1

            # 4) age increases by one quarter
            age += VISIT_LEN_YEARS

    df = pd.DataFrame(records)
    if save_csv:
        if os.path.exists(csv_path):
            os.remove(csv_path)
        df.to_csv(csv_path, index=False)
        print(f"Saved '{csv_path}' ({len(df)} rows).")
    return df


# -------- Run once and basic sanity checks --------
if __name__ == "__main__":
    df = simulate_data(n_patients=500, n_visits=N_VISITS, seed=42, save_csv=True)

    print("\n--- Sanity checks ---")
    print("Columns:", df.columns.tolist())
    print("Patients:", df['patient_id'].nunique(), "| Max visit:", int(df['visit'].max()))
    print("Severe tox events:", int(df['severe_tox'].sum()))
    print("Deaths from toxicity:", int(df['death_tox'].sum()))
    print("Patients with any death:", int(df.groupby('patient_id')['death_tox'].max().sum()))
    print("Last-visit distribution (top):")
    print(df.groupby('patient_id')['visit'].max().value_counts().head())

    # Quick mapping preview (first 5 QOL rows with year labels)
    preview = (
        df[df['visit'].isin(QOL_VISITS)]
          .assign(year=lambda d: d['visit'].map(VISIT_TO_YEARS))
    )[["patient_id", "visit", "year", "QOL"]].head()
    print("\nPreview QOL rows:\n", preview)



In [3]:
# ============================
# Quarterly Simulation (40 visits = 10 years)
# ============================

import os
import numpy as np
import pandas as pd

# -------- Global setup (QUARTERLY) --------
VISIT_LEN_YEARS = 0.25          # 1 visit = quarter
N_VISITS = 40                   # 10 years → 40 quarters (1..40)
QOL_VISITS = set(range(4, 41, 4))  # annual QOL at quarters: 4,8,12,...,40
VISIT_TO_YEARS = {
    v: f"{v // 4} year" + ("s" if v // 4 > 1 else "")
    for v in sorted(QOL_VISITS)
}

# -------- Tunable parameters (REBALANCED) --------
AGE_PENALTY_PER_YEAR = 0.015    # QOL penalty per year >75
NOISE_SIGMA = 0.03              # random QOL noise

# Dose burden on QOL (independent of efficacy)
DOSE_PEN_FULL = (0.040, 0.060)  # (fit, frail) penalty when on full dose
DOSE_PEN_REDUCED = 0.006        # small reduced-burden

# Efficacy boost only when disease activity (clinical_test=1)
EFFICACY_FULL = 0.015           # was 0.030
EFFICACY_REDUCED = 0.008        # was 0.010

# Toxicity → immediate drop + partial, slower recovery (carry-over)
SEVERE_TOX_QOL_HIT_FIT = 0.15        # immediate QOL hit if fit
SEVERE_TOX_QOL_HIT_FRAIL = 0.22      # immediate QOL hit if frail
CARRYOVER_DAMAGE_FRACTION = 0.80     # fraction of hit that persists as "scar"
QOL_DAMAGE_RECOVERY_PER_QUARTER = 0.20  # heals 20% of stored damage per quarter

# === cumulative full-dose exposure (chronic toxicity, γραμμικό) ===
# cum_dose_cat: συνεχές (float) ≈ "πόσες full-dose επισκέψεις έχει φάει"
CUM_DOSE_SLOPE = 0.005                # +0.5% στο P(test=1) ανά full-dose visit
MAX_CUM_DOSE_EFFECT = 0.25            # cap στο +25% επιπλέον
CUM_DOSE_RECOVERY_PER_QUARTER = 0.10  # ανά quarter σε reduced/0 μειώνεται λίγο

def update_cum_dose_cat(cum_dose_cat, dose):
    """
    cum_dose_cat: συνεχές μέγεθος (float) που μετρά cumulative full-dose exposure.

    - Αν dose == 1 (full): προσθέτουμε 1.0 μονάδα (ένα ακόμα full quarter).
    - Αν dose == 0 (reduced/stop): αφαιρούμε CUM_DOSE_RECOVERY_PER_QUARTER (μέχρι min 0).
    """
    if dose == 1:
        cum_dose_cat = cum_dose_cat + 1.0
    else:
        cum_dose_cat = max(0.0, cum_dose_cat - CUM_DOSE_RECOVERY_PER_QUARTER)
    return float(cum_dose_cat)

def cum_dose_effect_on_p_pos(cum_dose_cat):
    """
    Γραμμική επίδραση της cumulative dose στην πιθανότητα bad test (test=1).
    """
    extra = CUM_DOSE_SLOPE * cum_dose_cat
    return float(np.clip(extra, 0.0, MAX_CUM_DOSE_EFFECT))


# --- Toxicity probabilities per QUARTER (relative risk reduced on reduced dose)
def p_toxicity_g34(age, dose):
    base = 0.12 if age < 65 else 0.17   # per quarter
    rr = 1.0 if dose == 1 else 0.55     # reduced = 45% risk reduction
    return float(np.clip(base * rr, 0, 1))


def p_toxicity_death(age, dose):
    base = 0.0015 if age < 65 else 0.0145  # per quarter Grade 5
    rr = 1.0 if dose == 1 else 0.50        # reduced halves the risk
    return float(np.clip(base * rr, 0, 1))


# -------- Clinical test evolution (disease control) --------
def base_next_test_positive_prob(current_test, dose, cum_dose_cat=0.0):
    """
    Softer dose effects + cumulative toxicity effect.
    """
    if current_test == 1:
        p_pos = 0.40
        p_pos *= 0.92 if dose == 1 else 1.03  # full slightly better; reduced slightly worse
    else:
        p_pos = 0.20
        p_pos *= 0.95 if dose == 1 else 1.05

    # chronic toxicity effect (μικρό, γραμμικό penalty ανά full visit)
    p_pos += cum_dose_effect_on_p_pos(cum_dose_cat)

    return float(np.clip(p_pos, 0.05, 0.99))


def next_test_positive_prob(current_test, dose, cum_dose_cat):
    """
    Χωρίς inertia: P(next=1) = base_next_test_positive_prob(...).
    """
    return base_next_test_positive_prob(current_test, dose, cum_dose_cat)


# -------- QOL generator (with efficacy & carry-over damage) --------
def simulate_qol(frailty, age, dose, clinical_test, rng, carryover_damage):
    """
    QOL ∈ [0,1], quarterly.
    """
    lo, hi = (0.55, 0.90) if frailty == 0 else (0.35, 0.70)
    base_qol = rng.uniform(lo, hi)
    age_pen = max(0.0, (age - 75) * AGE_PENALTY_PER_YEAR)

    # dose burden by frailty/dose
    dose_pen = (
        DOSE_PEN_FULL[0] if (dose == 1 and frailty == 0) else
        DOSE_PEN_FULL[1] if (dose == 1 and frailty == 1) else
        DOSE_PEN_REDUCED
    )

    # efficacy only if current test is positive
    efficacy = (EFFICACY_FULL if dose == 1 else EFFICACY_REDUCED) if clinical_test == 1 else 0.0

    noise = rng.normal(0, NOISE_SIGMA)
    qol = base_qol - age_pen - dose_pen + efficacy + noise - carryover_damage
    return float(np.clip(qol, 0, 1))


# -------- Frailty dynamics (less dose-driven) --------
def frailty_transition(frail, age, dose, clinical_test, pos_streak, rng):
    """
    Slower transitions with modest dose effects.
    """
    if frail == 0:
        p = 0.014 if age <= 75 else (0.022 if age <= 80 else 0.036)
        if dose == 1:
            p -= 0.003  # smaller protection from full
        if pos_streak >= 2:  # sustained activity raises risk
            p += 0.008 if dose == 0 else 0.003
        return 1 if rng.random() < np.clip(p, 0, 1) else 0
    else:
        r = 0.010 if age <= 75 else (0.008 if age <= 80 else 0.006)
        if dose == 1:
            r += 0.004  # smaller recovery boost on full
        if clinical_test == 0:
            r += 0.003  # good tests help recovery
        return 0 if rng.random() < np.clip(r, 0, 1) else 1


# -------- Initial dose policy --------
def assign_initial_dose(frail, rng):
    # fit → full, frail → 50/50
    return 1 if frail == 0 else int(rng.random() < 0.5)


# -------- Main simulator --------
def simulate_data(
    n_patients=500,
    n_visits=N_VISITS,
    seed=42,
    save_csv=True,
    csv_path='Fixed_patient_data.csv'
):
    rng = np.random.default_rng(seed)
    ids = np.arange(1, n_patients + 1)
    ages = rng.integers(71, 96, size=n_patients)  # starting ages
    frail0 = rng.choice([1, 0], size=n_patients, p=[0.7, 0.3])
    dose0 = np.array([assign_initial_dose(f, rng) for f in frail0])
    test0 = rng.integers(0, 2, size=n_patients)  # initial clinical test (0/1)

    print(
        f"Baseline: {np.mean(frail0 == 0) * 100:.1f}% fit / "
        f"{np.mean(frail0 == 1) * 100:.1f}% frail (n={n_patients})"
    )

    records = []

    for pid, age0, frail_init, dose_init, test_init in zip(ids, ages, frail0, dose0, test0):
        age = float(age0)
        frail = int(frail_init)
        dose = int(dose_init)
        test = int(test_init)
        alive = True
        cum_qaly = 0.0

        # streaks μόνο για frailty dynamics
        pos_streak = 1 if test == 1 else 0
        neg_streak = 1 if test == 0 else 0

        # persistent toxicity damage that heals gradually
        carryover_damage = 0.0

        # cumulative full-dose exposure counter
        cum_dose_cat = 0.0

        for visit_idx in range(n_visits):  # 0..39
            visit = visit_idx + 1          # 1..40
            if not alive:
                break

            # heal part of the stored damage each quarter
            carryover_damage *= (1.0 - QOL_DAMAGE_RECOVERY_PER_QUARTER)

            # QOL before mortality check (includes carry-over damage)
            qol = simulate_qol(frail, age, dose, test, rng, carryover_damage)

            # Toxicity events (per quarter)
            died_tox = rng.random() < p_toxicity_death(age, dose)
            severe_tox = False
            if not died_tox:
                severe_tox = rng.random() < p_toxicity_g34(age, dose)
                if severe_tox:
                    hit = SEVERE_TOX_QOL_HIT_FRAIL if frail == 1 else SEVERE_TOX_QOL_HIT_FIT
                    # immediate drop + store a fraction as "scar"
                    qol = max(0.0, qol - hit)
                    carryover_damage += hit * CARRYOVER_DAMAGE_FRACTION

            # QALYs accrue per quarter
            qaly = qol * VISIT_LEN_YEARS
            cum_qaly += qaly

            records.append({
                'patient_id': pid,
                'age': round(age, 2),
                'visit': visit,                      # 1..40
                'frailty': frail,
                'clinical_test': int(test),
                'full_dose': int(dose),
                'cum_dose_cat': float(cum_dose_cat),  # cumulative exposure state
                'QOL': qol if visit in QOL_VISITS else np.nan,
                'qaly_quarter': qaly,
                'cum_qaly': cum_qaly,
                'severe_tox': int(severe_tox),
                'death_tox': int(died_tox)
            })

            if died_tox:
                alive = False
                break

            # ---- evolve state/time for next quarter ----

            # 0) update cumulative dose *μετά* από αυτό το quarter (ανάλογα με το dose)
            cum_dose_cat = update_cum_dose_cat(cum_dose_cat, dose)

            # 1) clinical test χωρίς inertia, αλλά με cumulative toxicity
            p_next_pos = next_test_positive_prob(
                current_test=test,
                dose=dose,
                cum_dose_cat=cum_dose_cat
            )
            test = 1 if rng.random() < p_next_pos else 0
            if test == 1:
                pos_streak += 1
                neg_streak = 0
            else:
                neg_streak += 1
                pos_streak = 0

            # 2) frailty transition (slow + depends on activity streak)
            frail = frailty_transition(frail, age, dose, test, pos_streak, rng)

            # 3) Dose: σταθερή στο natural-history simulation
            # (οι policies θα αλλάζουν τη δόση σε επόμενο στάδιο αν θέλουμε)

            # 4) age increases by one quarter
            age += VISIT_LEN_YEARS

    df = pd.DataFrame(records)
    if save_csv:
        if os.path.exists(csv_path):
            os.remove(csv_path)
        df.to_csv(csv_path, index=False)
        print(f"Saved '{csv_path}' ({len(df)} rows).")
    return df


# -------- Run once and basic sanity checks --------
if __name__ == "__main__":
    df = simulate_data(n_patients=500, n_visits=N_VISITS, seed=42, save_csv=True)

    print("\n--- Sanity checks ---")
    print("Columns:", df.columns.tolist())
    print("Patients:", df['patient_id'].nunique(), "| Max visit:", int(df['visit'].max()))
    print("Severe tox events:", int(df['severe_tox'].sum()))
    print("Deaths from toxicity:", int(df['death_tox'].sum()))
    print("Patients with any death:", int(df.groupby('patient_id')['death_tox'].max().sum()))
    print("Last-visit distribution (top):")
    print(df.groupby('patient_id')['visit'].max().value_counts().head())

    # Quick mapping preview (first 5 QOL rows with year labels)
    preview = (
        df[df['visit'].isin(QOL_VISITS)]
          .assign(year=lambda d: d['visit'].map(VISIT_TO_YEARS))
    )[["patient_id", "visit", "year", "QOL"]].head()
    print("\nPreview QOL rows:\n", preview)



In [4]:
# ============================
# Quarterly Simulation (40 visits = 10 years)
# ============================

import os
import numpy as np
import pandas as pd

# -------- Global setup (QUARTERLY) --------
VISIT_LEN_YEARS = 0.25          # 1 visit = quarter
N_VISITS = 40                   # 10 years → 40 quarters (1..40)

# baseline (visit=0) + annual QOL at quarters: 4,8,12,...,40
QOL_VISITS = {0} | set(range(4, 41, 4))
VISIT_TO_YEARS = {
    0: "baseline",
    **{v: f"{v // 4} year" + ("s" if v // 4 > 1 else "")
       for v in sorted(QOL_VISITS) if v != 0}
}

# -------- Tunable parameters (REBALANCED) --------
AGE_PENALTY_PER_YEAR = 0.015    # QOL penalty per year >75
NOISE_SIGMA = 0.03              # random QOL noise

# Dose burden on QOL (independent of efficacy)
DOSE_PEN_FULL = (0.040, 0.060)  # (fit, frail) penalty when on full dose
DOSE_PEN_REDUCED = 0.006        # small reduced-burden

# Efficacy boost only when disease activity (clinical_test=1)
EFFICACY_FULL = 0.015           # was 0.030
EFFICACY_REDUCED = 0.008        # was 0.010

# Toxicity → immediate drop + partial, slower recovery (carry-over)
SEVERE_TOX_QOL_HIT_FIT = 0.15        # immediate QOL hit if fit
SEVERE_TOX_QOL_HIT_FRAIL = 0.22      # immediate QOL hit if frail
CARRYOVER_DAMAGE_FRACTION = 0.80     # fraction of hit that persists as "scar"
QOL_DAMAGE_RECOVERY_PER_QUARTER = 0.20  # heals 20% of stored damage per quarter

# === cumulative full-dose exposure (chronic toxicity, γραμμικό) ===
# cum_dose_cat: συνεχές (float) ≈ "πόσες full-dose επισκέψεις έχει φάει"
CUM_DOSE_SLOPE = 0.05                # +0.5% στο P(test=1) ανά full-dose visit
MAX_CUM_DOSE_EFFECT = 0.25            # cap στο +25% επιπλέον
CUM_DOSE_RECOVERY_PER_QUARTER = 0.10  # ανά quarter σε reduced/0 μειώνεται λίγο

def update_cum_dose_cat(cum_dose_cat, dose):
    """
    cum_dose_cat: συνεχές μέγεθος (float) που μετρά cumulative full-dose exposure.

    - Αν dose == 1 (full): προσθέτουμε 1.0 μονάδα (ένα ακόμα full quarter).
    - Αν dose == 0 (reduced/stop): αφαιρούμε CUM_DOSE_RECOVERY_PER_QUARTER (μέχρι min 0).
    """
    if dose == 1:
        cum_dose_cat = cum_dose_cat + 1.0
    else:
        cum_dose_cat = max(0.0, cum_dose_cat - CUM_DOSE_RECOVERY_PER_QUARTER)
    return float(cum_dose_cat)

def cum_dose_effect_on_p_pos(cum_dose_cat):
    """
    Γραμμική επίδραση της cumulative dose στην πιθανότητα bad test (test=1).
    """
    extra = CUM_DOSE_SLOPE * cum_dose_cat
    return float(np.clip(extra, 0.0, MAX_CUM_DOSE_EFFECT))


# --- Toxicity probabilities per QUARTER (relative risk reduced on reduced dose)
def p_toxicity_g34(age, dose):
    base = 0.12 if age < 65 else 0.17   # per quarter
    rr = 1.0 if dose == 1 else 0.55     # reduced = 45% risk reduction
    return float(np.clip(base * rr, 0, 1))


def p_toxicity_death(age, dose):
    base = 0.0015 if age < 65 else 0.0145  # per quarter Grade 5
    rr = 1.0 if dose == 1 else 0.50        # reduced halves the risk
    return float(np.clip(base * rr, 0, 1))


# -------- Clinical test evolution (disease control) --------
def base_next_test_positive_prob(current_test, dose, cum_dose_cat=0.0):
    """
    Softer dose effects + cumulative toxicity effect.
    """
    if current_test == 1:
        p_pos = 0.40
        p_pos *= 0.92 if dose == 1 else 1.03  # full slightly better; reduced slightly worse
    else:
        p_pos = 0.20
        p_pos *= 0.95 if dose == 1 else 1.05

    # chronic toxicity effect (μικρό, γραμμικό penalty ανά full visit)
    p_pos += cum_dose_effect_on_p_pos(cum_dose_cat)

    return float(np.clip(p_pos, 0.05, 0.99))


def next_test_positive_prob(current_test, dose, cum_dose_cat):
    """
    Χωρίς extra inertia: P(next=1) = base_next_test_positive_prob(...).
    """
    return base_next_test_positive_prob(current_test, dose, cum_dose_cat)


# -------- Baseline QOL (πριν επηρεάσει η δόση) --------
def simulate_qol_baseline(frailty, age, rng):
    """
    Baseline QOL στο t=0, πριν από επίδραση δόσης / efficacy / toxicity.
    Εξαρτάται μόνο από frailty, ηλικία και noise.
    """
    lo, hi = (0.6, 0.90) if frailty == 0 else (0.4, 0.70)
    base_qol = rng.uniform(lo, hi)
    age_pen  = max(0.0, (age - 75) * AGE_PENALTY_PER_YEAR)
    noise    = rng.normal(0, NOISE_SIGMA)
    qol = base_qol - age_pen + noise
    return float(np.clip(qol, 0, 1))


# -------- QOL generator (with efficacy & carry-over damage) --------
def simulate_qol(frailty, age, dose, clinical_test, rng, carryover_damage):
    """
    QOL ∈ [0,1], quarterly, ΜΕΤΑ την έναρξη θεραπείας.
    """
    lo, hi = (0.6, 0.90) if frailty == 0 else (0.4, 0.70)
    base_qol = rng.uniform(lo, hi)
    age_pen = max(0.0, (age - 75) * AGE_PENALTY_PER_YEAR)

    # dose burden by frailty/dose
    dose_pen = (
        DOSE_PEN_FULL[0] if (dose == 1 and frailty == 0) else
        DOSE_PEN_FULL[1] if (dose == 1 and frailty == 1) else
        DOSE_PEN_REDUCED
    )

    # efficacy only if current test is positive
    efficacy = (EFFICACY_FULL if dose == 1 else EFFICACY_REDUCED) if clinical_test == 1 else 0.0

    noise = rng.normal(0, NOISE_SIGMA)
    qol = base_qol - age_pen - dose_pen + efficacy + noise - carryover_damage
    return float(np.clip(qol, 0, 1))


# -------- Frailty dynamics (less dose-driven) --------
def frailty_transition(frail, age, dose, clinical_test, pos_streak, rng):
    """
    Slower transitions with modest dose effects.
    """
    if frail == 0:
        p = 0.014 if age <= 75 else (0.022 if age <= 80 else 0.036)
        if dose == 1:
            p -= 0.003  # smaller protection from full
        if pos_streak >= 2:  # sustained activity raises risk
            p += 0.008 if dose == 0 else 0.003
        return 1 if rng.random() < np.clip(p, 0, 1) else 0
    else:
        r = 0.010 if age <= 75 else (0.008 if age <= 80 else 0.006)
        if dose == 1:
            r += 0.004  # smaller recovery boost on full
        if clinical_test == 0:
            r += 0.003  # good tests help recovery
        return 0 if rng.random() < np.clip(r, 0, 1) else 1


# -------- Initial dose policy --------
def assign_initial_dose(frail, rng):
    # fit → full, frail → 50/50
    return 1 if frail == 0 else int(rng.random() < 0.5)


# -------- Main simulator --------
def simulate_data(
    n_patients=500,
    n_visits=N_VISITS,
    seed=42,
    save_csv=True,
    csv_path='Fixed_patient_data.csv'
):
    rng = np.random.default_rng(seed)
    ids = np.arange(1, n_patients + 1)
    ages = rng.integers(71, 96, size=n_patients)  # starting ages
    frail0 = rng.choice([1, 0], size=n_patients, p=[0.7, 0.3])
    dose0 = np.array([assign_initial_dose(f, rng) for f in frail0])
    test0 = rng.integers(0, 2, size=n_patients)  # initial clinical test (0/1)

    print(
        f"Baseline: {np.mean(frail0 == 0) * 100:.1f}% fit / "
        f"{np.mean(frail0 == 1) * 100:.1f}% frail (n={n_patients})"
    )

    records = []

    for pid, age0, frail_init, dose_init, test_init in zip(ids, ages, frail0, dose0, test0):
        age = float(age0)
        frail = int(frail_init)
        dose = int(dose_init)
        test = int(test_init)
        alive = True
        cum_qaly = 0.0

        # streaks μόνο για frailty dynamics
        pos_streak = 1 if test == 1 else 0
        neg_streak = 1 if test == 0 else 0

        # persistent toxicity damage that heals gradually
        carryover_damage = 0.0

        # cumulative full-dose exposure counter
        cum_dose_cat = 0.0

        # ----- Baseline visit (t = 0, πριν από επίδραση δόσης) -----
        baseline_qol = simulate_qol_baseline(frail, age, rng)
        records.append({
            'patient_id': pid,
            'age': round(age, 2),
            'visit': 0,                     # baseline
            'frailty': frail,
            'clinical_test': int(test),
            'full_dose': int(dose),         # starting dose (για definition cohorts)
            'cum_dose_cat': float(cum_dose_cat),  # 0.0
            'QOL': baseline_qol,
            'qaly_quarter': 0.0,            # baseline snapshot → 0 QALY
            'cum_qaly': 0.0,
            'severe_tox': 0,
            'death_tox': 0
        })

        # ----- Treatment quarters 1..n_visits -----
        for visit_idx in range(n_visits):  # 0..39
            visit = visit_idx + 1          # 1..40
            if not alive:
                break

            # heal part of the stored damage each quarter
            carryover_damage *= (1.0 - QOL_DAMAGE_RECOVERY_PER_QUARTER)

            # QOL before mortality check (includes carry-over damage)
            qol = simulate_qol(frail, age, dose, test, rng, carryover_damage)

            # Toxicity events (per quarter)
            died_tox = rng.random() < p_toxicity_death(age, dose)
            severe_tox = False
            if not died_tox:
                severe_tox = rng.random() < p_toxicity_g34(age, dose)
                if severe_tox:
                    hit = SEVERE_TOX_QOL_HIT_FRAIL if frail == 1 else SEVERE_TOX_QOL_HIT_FIT
                    # immediate drop + store a fraction as "scar"
                    qol = max(0.0, qol - hit)
                    carryover_damage += hit * CARRYOVER_DAMAGE_FRACTION

            # QALYs accrue per quarter
            qaly = qol * VISIT_LEN_YEARS
            cum_qaly += qaly

            records.append({
                'patient_id': pid,
                'age': round(age, 2),
                'visit': visit,                      # 1..40
                'frailty': frail,
                'clinical_test': int(test),
                'full_dose': int(dose),
                'cum_dose_cat': float(cum_dose_cat),  # cumulative exposure state (πριν update)
                'QOL': qol if visit in QOL_VISITS else np.nan,
                'qaly_quarter': qaly,
                'cum_qaly': cum_qaly,
                'severe_tox': int(severe_tox),
                'death_tox': int(died_tox)
            })

            if died_tox:
                alive = False
                break

            # ---- evolve state/time for next quarter ----

            # 0) update cumulative dose *μετά* από αυτό το quarter (ανάλογα με το dose)
            cum_dose_cat = update_cum_dose_cat(cum_dose_cat, dose)

            # 1) clinical test χωρίς inertia, αλλά με cumulative toxicity
            p_next_pos = next_test_positive_prob(
                current_test=test,
                dose=dose,
                cum_dose_cat=cum_dose_cat
            )
            test = 1 if rng.random() < p_next_pos else 0
            if test == 1:
                pos_streak += 1
                neg_streak = 0
            else:
                neg_streak += 1
                pos_streak = 0

            # 2) frailty transition (slow + depends on activity streak)
            frail = frailty_transition(frail, age, dose, test, pos_streak, rng)

            # 3) Dose: σταθερή στο natural-history simulation
            # (οι policies θα αλλάζουν τη δόση σε επόμενο στάδιο αν θέλουμε)

            # 4) age increases by one quarter
            age += VISIT_LEN_YEARS

    df = pd.DataFrame(records)
    if save_csv:
        if os.path.exists(csv_path):
            os.remove(csv_path)
        df.to_csv(csv_path, index=False)
        print(f"Saved '{csv_path}' ({len(df)} rows).")
    return df


# -------- Run once and basic sanity checks --------
if __name__ == "__main__":
    df = simulate_data(n_patients=500, n_visits=N_VISITS, seed=42, save_csv=True)

    print("\n--- Sanity checks ---")
    print("Columns:", df.columns.tolist())
    print("Patients:", df['patient_id'].nunique(), "| Max visit:", int(df['visit'].max()))
    print("Severe tox events:", int(df['severe_tox'].sum()))
    print("Deaths from toxicity:", int(df['death_tox'].sum()))
    print("Patients with any death:", int(df.groupby('patient_id')['death_tox'].max().sum()))
    print("Last-visit distribution (top):")
    print(df.groupby('patient_id')['visit'].max().value_counts().head())

    # Quick mapping preview (πρώτες 5 QOL γραμμές με labels)
    preview = (
        df[df['visit'].isin(QOL_VISITS)]
          .assign(year=lambda d: d['visit'].map(VISIT_TO_YEARS))
    )[["patient_id", "visit", "year", "QOL"]].head()
    print("\nPreview QOL rows:\n", preview)

In [5]:
# --- Ensure df exists (load or simulate) ---
try:
    df  # just to check if it's defined
except NameError:
    import os, pandas as pd
    if os.path.exists('Fixed_patient_data.csv'):
        df = pd.read_csv('Fixed_patient_data.csv')
        print("Loaded 'Fixed_patient_data.csv'.")
    else:
        # assumes simulate_data(...) is already defined in this session
        df = simulate_data(n_patients=500, n_visits=40, seed=42, save_csv=True)
        print("Simulated data and saved to 'Fixed_patient_data.csv'.")

# --- Basic type hygiene (helps with pivots/groupbys) ---
for col in ['visit','frailty','full_dose']:
    if col in df.columns:
        df[col] = df[col].astype(int)

# Optional: check required columns exist
required_cols = {'patient_id','visit','frailty','full_dose'}
missing = required_cols - set(df.columns)
assert not missing, f"Missing columns: {missing}"

import numpy as np
import pandas as pd

# --- Βοηθητικά ---
def _pct(num, den):
    return 100.0 * num / den if den else np.nan

def _safe_mean(s):
    s = pd.Series(s).dropna()
    return float(s.mean()) if len(s) else np.nan

# --- 1) Visit 0 Snapshot ---
v0 = df.loc[df['visit'] == 0].copy()
total0 = len(v0)
fit_v0 = v0[v0['frailty'] == 0]
frail_v0 = v0[v0['frailty'] == 1]

print(f"Visit 0 — Total: {total0}, Fit: {len(fit_v0)} ({_pct(len(fit_v0), total0):.1f}%), "
      f"Frail: {len(frail_v0)} ({_pct(len(frail_v0), total0):.1f}%)")

# Δόσεις στο visit 0
fit_full = int(fit_v0['full_dose'].sum())
fit_red  = len(fit_v0) - fit_full
frail_full = int(frail_v0['full_dose'].sum())
frail_red  = len(frail_v0) - frail_full

print("\nInitial dose distribution (Visit 0):")
print(f"  Fit:   full={fit_full} ({_pct(fit_full, len(fit_v0)):.1f}%), "
      f"reduced={fit_red} ({_pct(fit_red, len(fit_v0)):.1f}%)")
print(f"  Frail: full={frail_full} ({_pct(frail_full, len(frail_v0)):.1f}%), "
      f"reduced={frail_red} ({_pct(frail_red, len(frail_v0)):.1f}%)")

# --- 1a) Assertions πολιτικής (για να σε προειδοποιούν αν κάτι «ξέφυγε») ---
# Fit πρέπει να είναι full στη βάση (το περιμένουμε 100%)
if len(fit_v0):
    assert fit_v0['full_dose'].min() == 1, \
        "Policy breach: Υπάρχουν FIT με reduced dose στο visit 0."

# Frail ~50/50 (+/- 10% ανοχή)
if len(frail_v0):
    frac_full_frail = frail_full / len(frail_v0)
    assert 0.40 <= frac_full_frail <= 0.60, \
        f"Policy skew: Frail full proportion={frac_full_frail:.2f} (εκτός 40–60%)."

# --- 2) Εξέλιξη δόσης ανά visit ---
n_visits = int(df['visit'].max() + 1)
print("\nDose evolution across visits:")
for v in range(n_visits):
    dv = df.loc[df['visit'] == v]
    full_cnt = int(dv['full_dose'].sum())
    print(f"  Visit {v:2d}: full={full_cnt} ({_pct(full_cnt, len(dv)):.1f}%)")

# --- 3) Πόσο συχνά «γυρίζει» η δόση (per patient) ---
dose_by_pid = df.pivot_table(index='patient_id', columns='visit', values='full_dose', aggfunc='first')
switches = (dose_by_pid.diff(axis=1).abs() == 1).sum(axis=1)  # # of 0↔1 transitions
print(f"\nDose switching (per patient): mean={switches.mean():.2f}, median={switches.median():.0f}, "
      f"90th pct={switches.quantile(0.9):.0f}")

# Κατηγοριοποίηση πολιτικής ανά ασθενή
prop_full = dose_by_pid.mean(axis=1)
policy_label = pd.Series(np.where(prop_full == 1.0, 'Always Full',
                        np.where(prop_full == 0.0, 'Always Reduced',
                        np.where(prop_full >= 0.67, 'Mostly Full',
                        np.where(prop_full <= 0.33, 'Mostly Reduced', 'Mixed')))),
                        index=prop_full.index)

policy_counts = policy_label.value_counts()
print("\nPatient policy buckets:")
for k, v in policy_counts.items():
    print(f"  {k}: {v}")

# --- 4) QOL διαγνωστικά στις επισκέψεις μέτρησης ---
qol_df = df[df['QOL'].notna()].copy()

# (α) Trajectory μέσης QOL ανά visit
traj = qol_df.groupby('visit')['QOL'].mean().reset_index(name='mean_QOL')
print("\nMean QOL on measurement visits (first 5 rows):")
print(traj.head())

# (β) QOL ανά frailty (την ώρα της μέτρησης)
qol_by_frailty = qol_df.groupby(['visit','frailty'])['QOL'].mean().unstack()
print("\nMean QOL by frailty at measurement visits (head):")
print(qol_by_frailty.head())

# (γ) QOL ανά «policy bucket» (στο visit της μέτρησης που υπάρχει)
# παίρνουμε τον μέσο όρο QOL κάθε ασθενή (στα visits που μετρήθηκαν) και τον συγκρίνουμε ανά bucket
mean_qol_per_pid = qol_df.groupby('patient_id')['QOL'].mean()
mean_qol_per_pid = mean_qol_per_pid.to_frame('mean_QOL').join(policy_label.rename('policy'))
summary_by_policy = mean_qol_per_pid.groupby('policy')['mean_QOL'].agg(['count','mean','std'])
print("\nMean QOL per patient by dose policy:")
print(summary_by_policy.sort_values('mean', ascending=False))

# --- 5) Toxicity προφίλ (αν έχεις πεδία severe_tox & death_tox από τη νέα προσομοίωση) ---
if {'severe_tox','death_tox'}.issubset(df.columns):
    tox_by_visit = df.groupby('visit')[['severe_tox','death_tox']].mean().rename(
        columns={'severe_tox':'severe_tox_rate','death_tox':'death_tox_rate'}).reset_index()
    cum_deaths = int(df.groupby('visit')['death_tox'].sum().cumsum().iloc[-1])
    print(f"\nToxicity summary: cumulative deaths from toxicity = {cum_deaths}")
    print("Severe & death toxicity rates by visit (head):")
    print(tox_by_visit.head())
else:
    print("\n(No toxicity columns present to summarize.)")

In [7]:
# ============================
# Cell 2 — QOL trajectories (quarterly → yearly) + N at risk
# ============================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Visits/Years mapping for quarterly data ---
qol_visits = list(range(4, 41, 4))  # 4,8,12,...,40  (1..10 years)
visit_to_year_index = {v: i+1 for i, v in enumerate(qol_visits)}  # 1..10
visit_to_year_label = {v: f"{i+1}" for i, v in enumerate(qol_visits)}  # labels "1","2",...,"10"
year_ticks = list(range(1, len(qol_visits)+1))
year_labels = [str(i) for i in year_ticks]

def traj_stats(df_subset, visits):
    """Return mean, sd, n, se, 95%CI for QOL at each visit (filtered to visits, dropna QOL)."""
    s = (df_subset[df_subset['visit'].isin(visits)]
         .dropna(subset=['QOL'])
         .groupby('visit', as_index=False)['QOL']
         .agg(mean='mean', std='std', n='count'))
    s['se'] = s['std'] / np.sqrt(s['n'].replace(0, np.nan))
    s['ci95'] = 1.96 * s['se']
    s['x'] = s['visit'].map(visit_to_year_index)
    return s.sort_values('x')

# --- ALL ---
stats_all = traj_stats(df, qol_visits)
plt.figure(figsize=(9, 5))
plt.errorbar(stats_all['x'], stats_all['mean'], yerr=stats_all['ci95'], fmt='-o', capsize=4, label='All')
plt.title("QOL Trajectories — All Patients")
plt.xlabel("Time (Years)")
plt.ylabel("QOL")
plt.xticks(ticks=year_ticks, labels=year_labels)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# --- FIT at that visit (frailty==0 at the same visit) ---
stats_fit = traj_stats(df[df['frailty'] == 0], qol_visits)
plt.figure(figsize=(9, 5))
plt.errorbar(stats_fit['x'], stats_fit['mean'], yerr=stats_fit['ci95'], fmt='-o', capsize=4, label='Fit (frailty=0)')
plt.title("QOL Trajectories — Fit at that visit")
plt.xlabel("Time (Years)")
plt.ylabel("QOL")
plt.xticks(ticks=year_ticks, labels=year_labels)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# --- FRAIL at that visit (frailty==1 at the same visit) ---
stats_frail = traj_stats(df[df['frailty'] == 1], qol_visits)
plt.figure(figsize=(9, 5))
plt.errorbar(stats_frail['x'], stats_frail['mean'], yerr=stats_frail['ci95'], fmt='-o', capsize=4, label='Frail (frailty=1)')
plt.title("QOL Trajectories — Frail at that visit")
plt.xlabel("Time (Years)")
plt.ylabel("QOL")
plt.xticks(ticks=year_ticks, labels=year_labels)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# --- N at risk per year (using last recorded year per patient) ---
# keep only yearly QOL rows
df_yearly = (df[df['visit'].isin(qol_visits)]
             .dropna(subset=['QOL'])
             .assign(year_idx=lambda d: d['visit'].map(visit_to_year_index))
             .copy())

# last year index with QOL per patient (1..10)
last_year_per_pid = (df_yearly.sort_values(['patient_id','year_idx'])
                                .groupby('patient_id')['year_idx'].max())

print("\nPatients with QOL recorded (at risk) per year:")
for y in year_ticks:
    n = int((last_year_per_pid >= y).sum())
    print(f"{str(y):>2s} year{'s' if y>1 else ' '}:  {n}")

# Optional: quick check that death_tox exists and there are events
if 'death_tox' in df.columns:
    total_deaths = int(df.groupby('patient_id')['death_tox'].max().sum())
    print("\nTotal patients with death_tox=1:", total_deaths)
else:
    print("\n(No 'death_tox' column found — endpoints are assumed censoring/missing.)")

In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Auto-detect quarterly year visits based on visit indexing ---
vmin = int(df['visit'].min())
vmax = int(df['visit'].max())

if vmin == 0:
    qol_visits = list(range(3, min(39, vmax) + 1, 4))  # 3,7,11,...,<=39
else:
    qol_visits = list(range(4, min(40, vmax), 4)) + ([40] if vmax >= 40 else [])  # 4,8,12,...,40

# Labels 1..N years
year_idx_map   = {v: i+1 for i, v in enumerate(qol_visits)}          # 1..len
year_label_map = {v: f"{i+1}" for i, v in enumerate(qol_visits)}     # "1","2",...

# --- Keep only yearly QOL rows; clean NaN ---
df_yearly = (
    df[df['visit'].isin(qol_visits)]
      .dropna(subset=['QOL'])
      .assign(year_idx=lambda d: d['visit'].map(year_idx_map),
              year_label=lambda d: d['visit'].map(year_label_map))
      .copy()
)

# --- (Optional) sample for cleaner spaghetti ---
max_lines = 150
rng = np.random.default_rng(123)
pids = df_yearly['patient_id'].unique()
if len(pids) > max_lines:
    sample_ids = rng.choice(pids, size=max_lines, replace=False)
    df_lines = df_yearly[df_yearly['patient_id'].isin(sample_ids)]
else:
    df_lines = df_yearly

# --- Population mean & 95% CI per year ---
g = df_yearly.groupby('year_idx')['QOL']
stats = pd.DataFrame({'mean': g.mean(), 'std': g.std(), 'n': g.count()}).reset_index()
stats['se'] = stats['std'] / np.sqrt(stats['n'].replace(0, np.nan))
stats['ci95'] = 1.96 * stats['se']

# --- Plot ---
plt.figure(figsize=(12, 6))

# 1) spaghetti
for pid, sub in df_lines.sort_values(['patient_id','year_idx']).groupby('patient_id'):
    plt.plot(sub['year_idx'], sub['QOL'], alpha=0.25, linewidth=1)

# 2) mean ± 95% CI
plt.errorbar(stats['year_idx'], stats['mean'], yerr=stats['ci95'], fmt='-o', capsize=4,
             label='Mean ±95% CI', linewidth=2)

plt.title("Individual Patient QOL Trajectories (Yearly)")
plt.xlabel("Year")
plt.ylabel("Quality of Life (QOL)")
plt.xticks(ticks=stats['year_idx'], labels=[str(x) for x in stats['year_idx']])
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [11]:
# ============================
# Cell 2 — QOL trajectories by BASELINE groups + N at risk
# ============================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Visits/Years mapping for quarterly data (QOL measured yearly) ---
qol_visits = list(range(4, 41, 4))  # 4,8,12,...,40  (1..10 years if visits start at 1)
visit_to_year_index = {v: i+1 for i, v in enumerate(qol_visits)}  # 1..10
year_ticks  = list(range(1, len(qol_visits)+1))
year_labels = [str(i) for i in year_ticks]

def traj_stats(df_subset, visits):
    """Return mean, sd, n, se, 95%CI for QOL at each visit."""
    s = (df_subset[df_subset['visit'].isin(visits)]
         .dropna(subset=['QOL'])
         .groupby('visit', as_index=False)['QOL']
         .agg(mean='mean', std='std', n='count'))
    if len(s) == 0:
        # empty guard
        return pd.DataFrame({'x': [], 'mean': [], 'ci95': [], 'n': []})
    s['se']   = s['std'] / np.sqrt(s['n'].replace(0, np.nan))
    s['ci95'] = 1.96 * s['se']
    s['x']    = s['visit'].map(visit_to_year_index)
    return s.sort_values('x')

# --- Define BASELINE (initial) visit robustly ---
initial_visit = int(df['visit'].min())   # works whether 0- or 1-indexed

# --- Build cohorts based on BASELINE status ---
v0 = df.loc[df['visit'] == initial_visit, ['patient_id','frailty','full_dose']].copy()

coh_all_ids          = set(v0['patient_id'])
coh_fit_full_ids     = set(v0[(v0['frailty']==0) & (v0['full_dose']==1)]['patient_id'])
coh_frail_full_ids   = set(v0[(v0['frailty']==1) & (v0['full_dose']==1)]['patient_id'])
coh_frail_red_ids    = set(v0[(v0['frailty']==1) & (v0['full_dose']==0)]['patient_id'])

cohorts = [
    ("All patients (baseline)", coh_all_ids),
    ("Fit + Full at baseline",  coh_fit_full_ids),
    ("Frail + Full at baseline",coh_frail_full_ids),
    ("Frail + Reduced at baseline", coh_frail_red_ids),
]

# --- Plot 4 panels with mean ±95% CI ---
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True)
axes = axes.ravel()

for ax, (title, pid_set) in zip(axes, cohorts):
    if len(pid_set) == 0:
        ax.set_title(f"{title}\n(n=0)"); ax.grid(True, alpha=0.3)
        ax.set_xlabel("Time (Years)"); ax.set_ylabel("QOL")
        ax.set_xticks(year_ticks); ax.set_xticklabels(year_labels); continue

    sub = df[df['patient_id'].isin(pid_set)].copy()
    s = traj_stats(sub, qol_visits)
    ax.errorbar(s['x'], s['mean'], yerr=s['ci95'], fmt='-o', capsize=4, label=f'{title}')
    ax.set_title(f"{title}  (n={len(pid_set)})")
    ax.set_xlabel("Time (Years)")
    ax.set_ylabel("QOL")
    ax.set_xticks(year_ticks)
    ax.set_xticklabels(year_labels)
    ax.grid(True, alpha=0.3)

plt.suptitle("QOL Trajectories by Baseline Cohorts", fontsize=14)
plt.tight_layout(rect=[0,0,1,0.96])
plt.show()

# --- N at risk per year, for EACH cohort ---
def at_risk_counts(df_in, pid_set):
    """# with QOL recorded at or beyond each year among baseline cohort."""
    if len(pid_set) == 0:
        return pd.Series({y: 0 for y in year_ticks})
    yearly = (df_in[df_in['visit'].isin(qol_visits)]
                .dropna(subset=['QOL'])
                .assign(year_idx=lambda d: d['visit'].map(visit_to_year_index)))
    yearly = yearly[yearly['patient_id'].isin(pid_set)]
    if yearly.empty:
        return pd.Series({y: 0 for y in year_ticks})
    last_year = (yearly.sort_values(['patient_id','year_idx'])
                        .groupby('patient_id')['year_idx'].max())
    return pd.Series({y: int((last_year >= y).sum()) for y in year_ticks})

print("\nPatients at risk (with QOL recorded) per year by baseline cohort:")
for title, pid_set in cohorts:
    counts = at_risk_counts(df, pid_set)
    counts.index = [f"{i}" for i in counts.index]
    print(f"\n{title} (n={len(pid_set)}):")
    print(counts.to_string())

# Optional: check deaths if available
if 'death_tox' in df.columns:
    deaths_total = int(df.groupby('patient_id')['death_tox'].max().sum())
    print("\nTotal patients with death_tox=1:", deaths_total)

In [12]:
import numpy as np
import pandas as pd
from collections import defaultdict

class SimpleQOLMDP:
    """
    QOL-based MDP με 8 states: (frailty ∈{0,1}, clinical_test ∈{0,1}, dose ∈{0,1})
    Actions: 0=reduced_dose, 1=full_dose
    """

    # ---------- CHUNK 1: init / setup ----------
    def __init__(self, csv_file='Fixed_patient_data.csv', annual_discount=0.97):
        print("="*80); print("SIMPLE QOL-BASED MDP — unified class"); print("="*80)
        self.df = pd.read_csv(csv_file)
        print(f"Loaded: {len(self.df)} rows, {self.df['patient_id'].nunique()} patients")

        must = ['patient_id','visit','frailty','clinical_test','full_dose']
        missing = [c for c in must if c not in self.df.columns]
        assert not missing, f"Missing columns: {missing}"
        for c in must: self.df[c] = self.df[c].astype(int)

        self.gamma = float(annual_discount**0.25)   # quarterly discount

        print(f"Quarterly gamma = {self.gamma:.5f}")



        self.setup_states_and_actions()

    def setup_states_and_actions(self):
        self.states = {}
        idx = 0
        for frailty in [0,1]:
            for test in [0,1]:
                for dose in [0,1]:
                    self.states[idx] = {
                        'frailty': frailty,
                        'clinical_test': test,
                        'dose': dose,
                        'name': f"{'Fit' if frailty==0 else 'Frail'}_"
                                f"{'Neg' if test==0 else 'Pos'}_"
                                f"{'Red' if dose==0 else 'Full'}"
                    }
                    idx += 1
        self.n_states = len(self.states)
        self.actions = {0:'reduced_dose', 1:'full_dose'}
        self.n_actions = len(self.actions)
        self.state_mapping = {(s['frailty'], s['clinical_test'], s['dose']): i
                              for i,s in self.states.items()}
        print("States:")
        for i,s in self.states.items():
            print(f"  {i}: {s['name']} (frailty={s['frailty']}, test={s['clinical_test']}, dose={s['dose']})")

    def get_state_index(self, frailty, clinical_test, dose):
        return self.state_mapping.get((int(frailty), int(clinical_test), int(dose)), None)

    def analyze_data_coverage(self):
        print("\n" + "="*60); print("DATA COVERAGE"); print("="*60)
        has_qol = 'QOL' in self.df.columns
        for i, st in self.states.items():
            m = (
                (self.df['frailty']==st['frailty']) &
                (self.df['clinical_test']==st['clinical_test']) &
                (self.df['full_dose']==st['dose'])
            )
            n = int(m.sum())
            n_qol = int(self.df.loc[m,'QOL'].notna().sum()) if has_qol else 0
            print(f"{st['name']:<15}: {n:4d} rows, {n_qol:4d} with QOL")

    # ---------- CHUNK 2: rewards ----------
    def estimate_qol_rewards(self, lookahead_steps=3, reward_mode="qaly", visit_len_years=0.25):
        """
        Φτιάχνει R[s,a] από δεδομένα.
        - transition-based με lookahead έως `lookahead_steps` visits
        - direct state QOL ως fallback
        - defaults αν λείπουν όλα
        reward_mode: 'qol' ή 'qaly' (QOL * visit_len_years)
        """
        print("\n" + "="*60); print("QOL-BASED REWARD ESTIMATION"); print("="*60)

        def _next_measured_qol(traj_df, start_idx, max_lookahead):
            n = len(traj_df)
            for k in range(1, max_lookahead+1):
                j = start_idx + k
                if j < n:
                    q = traj_df.iloc[j]['QOL']
                    if pd.notna(q):
                        return float(q), k
            return None, None

        # Method 1: direct QOL per state
        print("\nDirect QOL by state:")
        qol_by_state = {}
        for s in range(self.n_states):
            st = self.states[s]
            mask = (
                (self.df['frailty']==st['frailty']) &
                (self.df['clinical_test']==st['clinical_test']) &
                (self.df['full_dose']==st['dose']) &
                (self.df['QOL'].notna())
            )
            vals = self.df.loc[mask, 'QOL'].astype(float).values
            if vals.size:
                qol_by_state[s] = {
                    'mean': float(vals.mean()),
                    'std':  float(vals.std(ddof=0)),
                    'count': int(vals.size),
                }
                print(f"  {st['name']:<15}: {qol_by_state[s]['mean']:.3f} ± {qol_by_state[s]['std']:.3f} (n={qol_by_state[s]['count']})")
            else:
                print(f"  {st['name']:<15}: No QOL data")

        # Method 2: transition-based with look-ahead
        print("\nQOL by (state, action) with look-ahead:")
        qol_by_sa = defaultdict(list)
        for pid in self.df['patient_id'].unique():
            traj = (self.df[self.df['patient_id']==pid]
                    .sort_values('visit').reset_index(drop=True))
            for i in range(len(traj)-1):
                cur, nxt = traj.iloc[i], traj.iloc[i+1]
                s = self.get_state_index(cur['frailty'], cur['clinical_test'], cur['full_dose'])
                if s is None: 
                    continue
                a = int(nxt['full_dose'])
                q = nxt['QOL'] if pd.notna(nxt['QOL']) else None
                if q is None:
                    q, _ = _next_measured_qol(traj, i, lookahead_steps)
                if q is not None:
                    reward = float(q) if reward_mode=="qol" else float(q)*float(visit_len_years)
                    qol_by_sa[(s,a)].append(reward)

        for (s,a), L in qol_by_sa.items():
            if L:
                m, sd, n = float(np.mean(L)), float(np.std(L, ddof=0)), len(L)
                unit = "QALY/visit" if reward_mode=="qaly" else "QOL"
                print(f"  {self.states[s]['name']:<15} → {self.actions[a]:<12}: {m:.3f} ± {sd:.3f} ({unit}, n={n})")

        # Compose R
        self.R = np.zeros((self.n_states, self.n_actions), dtype=float)
        print("\nFinal Reward Matrix ({})".format("QALY/visit" if reward_mode=="qaly" else "QOL"))
        for s in range(self.n_states):
            for a in range(self.n_actions):
                if (s,a) in qol_by_sa and len(qol_by_sa[(s,a)])>0:
                    avg, src = float(np.mean(qol_by_sa[(s,a)])), "transition"
                elif s in qol_by_state:
                    base = qol_by_state[s]['mean']
                    avg, src = (base if reward_mode=="qol" else base*visit_len_years), "direct"
                else:
                    st = self.states[s]
                    if st['frailty']==0: base = 0.75 if a==1 else 0.65
                    else:                base = 0.45 if a==0 else 0.35
                    avg, src = (base if reward_mode=="qol" else base*visit_len_years), "default"
                self.R[s,a] = avg
                print(f"  {self.states[s]['name']:<15} + {self.actions[a]:<12}: {avg:.3f} ({src})")

        print("\nReward range:", f"{self.R.min():.3f} .. {self.R.max():.3f}")
        return None, None  # κρατάμε απλό API

    # ---------- CHUNK 3: transitions ----------
    def estimate_transition_probabilities(self, alpha=0.5):
        """
        P[s,a,s'] από διαδοχικές επισκέψεις (state_t, action_{t+1}, state_{t+1}),
        με Dirichlet smoothing alpha.
        """
        print("\n" + "="*60); print("TRANSITION PROBABILITY ESTIMATION"); print("="*60)
        has_death = 'death_tox' in self.df.columns
        counts = defaultdict(lambda: defaultdict(lambda: defaultdict(int)))
        total = 0

        for pid in self.df['patient_id'].unique():
            traj = (self.df[self.df['patient_id']==pid]
                    .sort_values('visit').reset_index(drop=True))
            n = len(traj)
            for i in range(n-1):
                cur, nxt = traj.iloc[i], traj.iloc[i+1]
                if has_death and int(cur.get('death_tox',0))==1:
                    break
                s  = self.get_state_index(cur['frailty'], cur['clinical_test'], cur['full_dose'])
                a  = int(nxt['full_dose'])
                s2 = self.get_state_index(nxt['frailty'], nxt['clinical_test'], nxt['full_dose'])
                if s is None or s2 is None: 
                    continue
                counts[s][a][s2] += 1
                total += 1

        print(f"Total transitions observed: {total}")
        self.P = np.zeros((self.n_states, self.n_actions, self.n_states), dtype=float)

        for s in range(self.n_states):
            for a in range(self.n_actions):
                vec = np.array([counts[s][a][s2] for s2 in range(self.n_states)], dtype=float)
                if vec.sum()>0:
                    probs = (vec + alpha) / (vec.sum() + alpha*self.n_states)
                else:
                    probs = np.full(self.n_states, 0.3/(self.n_states-1)); probs[s]=0.7
                self.P[s,a,:] = probs

        print("\nTransition Summary (top 3 when n>5):")
        for s in range(self.n_states):
            for a in range(self.n_actions):
                n_sa = int(sum(counts[s][a].values()))
                if n_sa > 5:
                    top = sorted([(s2, self.P[s,a,s2]) for s2 in range(self.n_states)],
                                 key=lambda x:x[1], reverse=True)[:3]
                    print(f"  {self.states[s]['name']:<15} + {self.actions[a]:<12} (n={n_sa}):")
                    for s2,p in top:
                        if p>0.10:
                            print(f"    → {self.states[s2]['name']:<15}: {p:.3f}")

    # ---------- CHUNK 4: solvers ----------
    def value_iteration(self, gamma=None, epsilon=1e-3, max_iterations=5000):
        gamma = self.gamma if gamma is None else float(gamma)
        print("\n" + "="*60); print(f"VALUE ITERATION (γ={gamma:.5f})"); print("="*60)
        V = np.zeros(self.n_states, dtype=float)
        policy = np.zeros(self.n_states, dtype=int)
        for it in range(max_iterations):
            V_old = V.copy()
            for s in range(self.n_states):
                Qa = np.array([self.R[s,a] + gamma*np.dot(self.P[s,a,:], V_old)
                               for a in range(self.n_actions)], dtype=float)
                V[s] = Qa.max()
                policy[s] = int(Qa.argmax())
            delta = float(np.max(np.abs(V - V_old)))
            if it % 100 == 0: print(f"  Iter {it:4d}: max Δ = {delta:.6f}")
            if delta < epsilon:
                print(f"Converged at iter {it+1} (max Δ = {delta:.6f})"); break
        else:
            print(f"Reached max iterations ({max_iterations}).")
        return V, policy

    def policy_evaluation(self, policy, gamma=None, epsilon=1e-3, max_iterations=5000):
        gamma = self.gamma if gamma is None else float(gamma)
        V = np.zeros(self.n_states, dtype=float)
        for it in range(max_iterations):
            delta = 0.0
            for s in range(self.n_states):   # Gauss–Seidel-like
                a = int(policy[s])
                Vs_old = V[s]
                V[s] = self.R[s,a] + gamma * np.dot(self.P[s,a,:], V)
                delta = max(delta, abs(V[s]-Vs_old))
            if it % 200 == 0: print(f"  PolicyEval {it:4d}: max Δ = {delta:.6f}")
            if delta < epsilon:
                print(f"  PolicyEval converged at {it+1} (max Δ = {delta:.6f})"); break
        else:
            print(f"  PolicyEval reached max iterations ({max_iterations}).")
        return V

    def policy_iteration(self, gamma=None, max_iterations=100, eval_epsilon=1e-3, eval_max_iter=5000):
        gamma = self.gamma if gamma is None else float(gamma)
        print("\n" + "="*60); print(f"POLICY ITERATION (γ={gamma:.5f})"); print("="*60)
        policy = np.argmax(self.R, axis=1).astype(int)  # sensible start
        for it in range(max_iterations):
            V = self.policy_evaluation(policy, gamma=gamma, epsilon=eval_epsilon, max_iterations=eval_max_iter)
            stable = True
            for s in range(self.n_states):
                Qa = np.array([self.R[s,a] + gamma*np.dot(self.P[s,a,:], V) for a in range(self.n_actions)])
                best_a = int(Qa.argmax())
                if best_a != policy[s]:
                    policy[s] = best_a
                    stable = False
            if stable:
                print(f"Policy converged after {it+1} improvement iterations."); break
        else:
            print(f"Reached maximum improvement iterations ({max_iterations}).")
        return V, policy

In [15]:
# Instantiate
mdp = SimpleQOLMDP('Fixed_patient_data.csv', annual_discount=0.97)
mdp.analyze_data_coverage()

# Rewards & Transitions
mdp.estimate_qol_rewards(lookahead_steps=3, reward_mode="qaly", visit_len_years=0.25)
mdp.estimate_transition_probabilities(alpha=0.5)

# Solve
V_vi, pol_vi = mdp.value_iteration()
V_pi, pol_pi = mdp.policy_iteration()

print("\nPolicies identical:", np.array_equal(pol_vi, pol_pi))
print("||V_vi - V_pi||:", np.linalg.norm(V_vi - V_pi))
print("Optimal policy (VI):", pol_vi)
print("State values (VI):", np.round(V_vi, 3))

In [16]:
"""
QOL Trajectory Analysis Extension for MDP (quarterly-consistent)
- Steps = visits (default 40 = 10 years)
- Rewards assumed QALY/visit αν το R έχει οριστεί έτσι
"""

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import pandas as pd

def simulate_trajectory(mdp, initial_state, policy, n_steps=40, gamma=None, rng=None):
    """
    Simulate a single patient trajectory following a given policy.
    Each step = one visit (quarter).
    Rewards are taken from mdp.R[s,a] (QOL or QALY/visit per your setup).
    """
    if gamma is None:
        gamma = mdp.gamma
    if rng is None:
        rng = np.random.default_rng()

    traj = {
        'states': [initial_state],
        'actions': [],
        'rewards': [],
        'qol_values': [],
        'cumulative_qol': [],
        'discounted_qol': []
    }

    s = initial_state
    cum = 0.0
    disc = 0.0

    for t in range(n_steps):
        a = int(policy[s])
        r = float(mdp.R[s, a])

        traj['actions'].append(a)
        traj['rewards'].append(r)
        traj['qol_values'].append(r)

        cum += r
        disc += r * (gamma ** t)
        traj['cumulative_qol'].append(cum)
        traj['discounted_qol'].append(disc)

        p_next = mdp.P[s, a, :]
        s = int(rng.choice(mdp.n_states, p=p_next))
        traj['states'].append(s)

    return traj

def simulate_multiple_trajectories(mdp, initial_state, policy, n_simulations=500, n_steps=40, gamma=None, seed=123):
    if gamma is None:
        gamma = mdp.gamma
    rng = np.random.default_rng(seed)

    all_traj = []
    for _ in range(n_simulations):
        # διαφορετικό rng state ανά simulation για ανεξαρτησία
        tr = simulate_trajectory(mdp, initial_state, policy, n_steps=n_steps, gamma=gamma, rng=rng)
        all_traj.append(tr)

    agg = {
        'mean_qol_over_time': [],
        'std_qol_over_time': [],
        'mean_cumulative_qol': [],
        'std_cumulative_qol': [],
        'mean_discounted_qol': [],
        'std_discounted_qol': [],
        'state_frequencies': defaultdict(list),
        'action_frequencies': defaultdict(list)
    }

    for t in range(n_steps):
        q_t = [tr['qol_values'][t] for tr in all_traj]
        agg['mean_qol_over_time'].append(float(np.mean(q_t)))
        agg['std_qol_over_time'].append(float(np.std(q_t)))

        c_t = [tr['cumulative_qol'][t] for tr in all_traj]
        agg['mean_cumulative_qol'].append(float(np.mean(c_t)))
        agg['std_cumulative_qol'].append(float(np.std(c_t)))

        d_t = [tr['discounted_qol'][t] for tr in all_traj]
        agg['mean_discounted_qol'].append(float(np.mean(d_t)))
        agg['std_discounted_qol'].append(float(np.std(d_t)))

        # states at step t
        states_t = [tr['states'][t] for tr in all_traj]
        for s in range(mdp.n_states):
            agg['state_frequencies'][s].append(float(np.mean(np.array(states_t)==s)))

        # actions at step t
        actions_t = [tr['actions'][t] for tr in all_traj]
        for a in range(mdp.n_actions):
            agg['action_frequencies'][a].append(float(np.mean(np.array(actions_t)==a)))

    return agg

def create_alternative_policies(mdp, optimal_policy=None):
    """
    Returns a dict of policies for comparison.
    If optimal_policy is None, compute via value iteration.
    """
    policies = {}

    # 1) Always reduced
    policies['always_reduced'] = np.zeros(mdp.n_states, dtype=int)

    # 2) Always full
    policies['always_full'] = np.ones(mdp.n_states, dtype=int)

    # 3) Frailty-based (reduced if frail, full if fit)
    policies['frailty_based'] = np.array([
        0 if mdp.states[s]['frailty'] == 1 else 1
        for s in range(mdp.n_states)
    ], dtype=int)

    # 4) Test-based (full if positive, reduced if negative)
    policies['clinical_test_based'] = np.array([
        1 if mdp.states[s]['clinical_test'] == 1 else 0
        for s in range(mdp.n_states)
    ], dtype=int)

    # 5) Random (fixed seed for reproducibility)
    rng = np.random.default_rng(42)
    policies['random'] = rng.integers(0, 2, size=mdp.n_states, dtype=int)

    # 6) Inverse-optimal (choose the worse action wrt Q)
    if optimal_policy is None:
        V_opt, optimal_policy = mdp.value_iteration()  # uses mdp.gamma
    else:
        V_opt, _ = mdp.value_iteration()  # ενημερωμένα V για Q-margins

    gamma = mdp.gamma
    inv = np.zeros(mdp.n_states, dtype=int)
    for s in range(mdp.n_states):
        Q = np.array([mdp.R[s, a] + gamma * np.dot(mdp.P[s, a, :], V_opt) for a in range(mdp.n_actions)])
        inv[s] = int(np.argmin(Q))
    policies['inverse_optimal'] = inv

    # Include optimal too for convenience
    policies['optimal'] = np.array(optimal_policy, dtype=int)

    return policies

def analyze_trajectories_by_initial_state(mdp, optimal_policy, alternative_policies, n_simulations=500, n_steps=40):
    """
    Simulate for all initial states and all policies.
    Returns nested dict: results[state_name][policy_name] -> aggregated stats.
    """
    results = {}
    for s in range(mdp.n_states):
        state_name = mdp.states[s]['name']
        results[state_name] = {}

        # Optimal
        results[state_name]['optimal'] = simulate_multiple_trajectories(
            mdp, s, optimal_policy, n_simulations=n_simulations, n_steps=n_steps, gamma=mdp.gamma
        )
        # Alternatives
        for pname, pol in alternative_policies.items():
            results[state_name][pname] = simulate_multiple_trajectories(
                mdp, s, pol, n_simulations=n_simulations, n_steps=n_steps, gamma=mdp.gamma
            )
    return results

def _aggregate_three_groups(trajectory_results, group_states, policy_name):
    """
    Helper: returns (mean, std) arrays across a list of state_names for a given policy_name.
    Handles missing keys safely.
    """
    # find minimal horizon across states present
    horizons = []
    series = []
    for st in group_states:
        if st in trajectory_results and policy_name in trajectory_results[st]:
            series.append(trajectory_results[st][policy_name]['mean_qol_over_time'])
            horizons.append(len(series[-1]))
    if not series:
        return None, None
    T = min(horizons)
    arr = np.array([s[:T] for s in series])
    return arr.mean(axis=0), arr.std(axis=0)

def visualize_trajectory_analysis(mdp, optimal_policy, trajectory_results, savefig='qol_trajectory_analysis.png'):
    """
    Create summary plots (fit / frail / all, QOL over time; cumulative summaries; state occupancy)
    with custom layout:
      - Top-left:   Frail aggregate
      - Bottom-left: Fit aggregate
      - Bottom-right: All aggregate
      - Top-middle:  Optimal advantage (from a canonical start)
      - Top-right:   State occupancy (optimal)
      - Bottom-middle: Final cumulative by type
    """
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    fig.suptitle('QOL Trajectory Analysis: Optimal vs Alternative Policies', fontsize=16, fontweight='bold')

    # assign slots
    ax_frail = axes[0, 0]   # top-left
    ax_fit   = axes[1, 0]   # bottom-left
    ax_all   = axes[1, 2]   # bottom-right
    ax_adv   = axes[0, 1]   # top-middle
    ax_occ   = axes[0, 2]   # top-right
    ax_cum   = axes[1, 1]   # bottom-middle

    # Colors per policy
    policy_colors = {
        'optimal': 'red',
        'always_reduced': 'gold',
        'always_full': 'blue',
        'frailty_based': 'green',
        'clinical_test_based': 'orange',
    }

    # Define groups
    fit_states   = [n for n in trajectory_results.keys() if n.startswith('Fit_')]
    frail_states = [n for n in trajectory_results.keys() if n.startswith('Frail_')]
    all_states   = list(trajectory_results.keys())

    # ---- Frail aggregate (TOP-LEFT)
    for pname, color in policy_colors.items():
        m, s = _aggregate_three_groups(trajectory_results, frail_states, pname)
        if m is None: 
            continue
        x = np.arange(len(m))
        ax_frail.plot(x, m, label=pname, linewidth=2, color=color, alpha=0.9)
        ax_frail.fill_between(x, m - s, m + s, alpha=0.2, color=color)
    ax_frail.set_title('Frail Patients — Mean QOL over visits')
    ax_frail.set_xlabel('Visit (quarter)'); ax_frail.set_ylabel('QOL or QALY/visit')
    ax_frail.grid(True, alpha=0.3); ax_frail.legend(fontsize=8)

    # ---- Fit aggregate (BOTTOM-LEFT)
    for pname, color in policy_colors.items():
        m, s = _aggregate_three_groups(trajectory_results, fit_states, pname)
        if m is None: 
            continue
        x = np.arange(len(m))
        ax_fit.plot(x, m, label=pname, linewidth=2, color=color, alpha=0.9)
        ax_fit.fill_between(x, m - s, m + s, alpha=0.2, color=color)
    ax_fit.set_title('Fit Patients — Mean QOL over visits')
    ax_fit.set_xlabel('Visit (quarter)'); ax_fit.set_ylabel('QOL or QALY/visit')
    ax_fit.grid(True, alpha=0.3); ax_fit.legend(fontsize=8)

    # ---- All aggregate (BOTTOM-RIGHT)
    for pname, color in policy_colors.items():
        m, s = _aggregate_three_groups(trajectory_results, all_states, pname)
        if m is None: 
            continue
        x = np.arange(len(m))
        ax_all.plot(x, m, label=pname, linewidth=2, color=color, alpha=0.9)
        ax_all.fill_between(x, m - s, m + s, alpha=0.2, color=color)
    ax_all.set_title('All Patients — Mean QOL over visits')
    ax_all.set_xlabel('Visit (quarter)'); ax_all.set_ylabel('QOL or QALY/visit')
    ax_all.grid(True, alpha=0.3); ax_all.legend(fontsize=8)

    # ---- Optimal advantage (TOP-MIDDLE)
    start_state = 'Fit_Neg_Red' if 'Fit_Neg_Red' in trajectory_results else list(trajectory_results.keys())[0]
    opt_c = trajectory_results[start_state]['optimal']['mean_cumulative_qol']
    for pname, color in policy_colors.items():
        if pname == 'optimal' or pname not in trajectory_results[start_state]:
            continue
        alt_c = trajectory_results[start_state][pname]['mean_cumulative_qol']
        T = min(len(opt_c), len(alt_c))
        adv = np.array(opt_c[:T]) - np.array(alt_c[:T])
        ax_adv.plot(np.arange(T), adv, color=color, linewidth=2, label=f'vs {pname}', alpha=0.9)
    ax_adv.axhline(0, ls='--', color='k', alpha=0.5)
    ax_adv.set_title(f'Optimal Advantage — start: {start_state}')
    ax_adv.set_xlabel('Visit (quarter)'); ax_adv.set_ylabel('QOL Advantage')
    ax_adv.grid(True, alpha=0.3); ax_adv.legend(fontsize=8)

    # ---- State occupancy (TOP-RIGHT)
    occ = trajectory_results[start_state]['optimal']['state_frequencies']
    for s in range(mdp.n_states):
        if s in occ:
            y = occ[s]
            ax_occ.plot(np.arange(len(y)), y, linewidth=2, label=mdp.states[s]['name'], alpha=0.9)
    ax_occ.set_title('State Occupancy (Optimal policy)')
    ax_occ.set_xlabel('Visit (quarter)'); ax_occ.set_ylabel('Probability')
    ax_occ.grid(True, alpha=0.3); ax_occ.legend(fontsize=8, ncol=2)

    # ---- Final cumulative by type (BOTTOM-MIDDLE)
    patient_types = ['Fit', 'Frail']
    for pname, color in policy_colors.items():
        fit_final, frail_final = [], []
        for st, res in trajectory_results.items():
            if pname in res:
                final = res[pname]['mean_cumulative_qol'][-1]
                (fit_final if st.startswith('Fit_') else frail_final).append(final)
        if fit_final and frail_final:
            x = [0, 1]
            y = [np.mean(fit_final), np.mean(frail_final)]
            ax_cum.plot(x, y, 'o-', color=color, linewidth=2, markersize=6, label=pname, alpha=0.9)
    ax_cum.set_xticks([0,1]); ax_cum.set_xticklabels(patient_types)
    ax_cum.set_title('Final Cumulative QOL by Type'); ax_cum.set_ylabel('Cumulative QOL')
    ax_cum.grid(True, alpha=0.3); ax_cum.legend(fontsize=8)

    plt.tight_layout()
    if savefig:
        plt.savefig(savefig, dpi=300, bbox_inches='tight')
    plt.show()


def print_trajectory_summary(trajectory_results, optimal_policy, alternative_policies):
    print("\n" + "="*80)
    print("TRAJECTORY ANALYSIS SUMMARY")
    print("="*80)

    # Average final cumulative across all starting states
    policy_names = ['optimal'] + [k for k in alternative_policies.keys() if k != 'optimal']
    perf = {}
    for pname in policy_names:
        finals = []
        for st, res in trajectory_results.items():
            if pname in res:
                finals.append(res[pname]['mean_cumulative_qol'][-1])
        if finals:
            finals = np.array(finals, dtype=float)
            perf[pname] = {'mean': finals.mean(), 'std': finals.std(), 'min': finals.min(), 'max': finals.max()}

    print(f"{'Policy':<20} {'Mean':>8} {'Std':>8} {'Min':>8} {'Max':>8}  {'Δ vs Optimal':>12}")
    print("-"*70)
    opt_mean = perf.get('optimal', {'mean':np.nan})['mean']
    for pname, stats in sorted(perf.items(), key=lambda x: x[1]['mean'], reverse=True):
        dv = stats['mean'] - opt_mean if np.isfinite(opt_mean) else np.nan
        print(f"{pname:<20} {stats['mean']:8.3f} {stats['std']:8.3f} {stats['min']:8.3f} {stats['max']:8.3f}  {dv:12.3f}")

    # By patient type
    print("\nBy patient type (final cumulative mean):")
    for tlabel in ['Fit', 'Frail']:
        row = {}
        for pname in policy_names:
            vals = []
            for st, res in trajectory_results.items():
                if st.startswith(tlabel) and pname in res:
                    vals.append(res[pname]['mean_cumulative_qol'][-1])
            if vals:
                row[pname] = float(np.mean(vals))
        if row:
            print(f"  {tlabel}: " + " | ".join([f"{k}:{v:.3f}" for k,v in sorted(row.items(), key=lambda x:x[1], reverse=True)]))

def add_trajectory_analysis(mdp, optimal_policy, n_simulations=500, n_steps=40):
    """
    Run end-to-end trajectory analysis and visuals.
    """
    print("\n" + "="*80)
    print("RUNNING TRAJECTORY ANALYSIS")
    print("="*80)

    # Build policy set
    print("Creating alternative policies...")
    alt_policies = create_alternative_policies(mdp, optimal_policy=optimal_policy)

    print(f"Simulating trajectories: {n_simulations} sims × {n_steps} visits × {mdp.n_states} start states × {len(alt_policies)} policies")
    results = analyze_trajectories_by_initial_state(
        mdp, optimal_policy, alt_policies, n_simulations=n_simulations, n_steps=n_steps
    )

    print("Creating visuals...")
    visualize_trajectory_analysis(mdp, optimal_policy, results)

    print_trajectory_summary(results, optimal_policy, alt_policies)

    print("\n" + "="*80)
    print("TRAJECTORY ANALYSIS COMPLETE!")
    print("="*80)
    return results

In [17]:
# --- helper: aggregate over an arbitrary list of state names for a given policy ---
def _aggregate_group(trajectory_results, state_names, policy_name):
    series = []
    T = None
    for st in state_names:
        if st in trajectory_results and policy_name in trajectory_results[st]:
            y = trajectory_results[st][policy_name]['mean_qol_over_time']
            series.append(np.array(y, dtype=float))
            T = len(y) if T is None else min(T, len(y))
    if not series:
        return None, None
    arr = np.vstack([s[:T] for s in series])
    return arr.mean(axis=0), arr.std(axis=0)

# --- helper: build baseline groups from mdp.states ---
def get_baseline_groups(mdp, trajectory_results):
    """
    Returns dict: {group_name: [state_names]} for the 4 baseline cohorts:
      - All patients
      - Fit + Full
      - Frail + Full
      - Frail + Reduced
    """
    # map index->name that exists in trajectory_results
    idx_to_name = {i: mdp.states[i]['name'] for i in range(mdp.n_states)
                   if mdp.states[i]['name'] in trajectory_results}
    groups = {
        "All patients (baseline)": list(idx_to_name.values()),
        "Fit + Full at baseline":  [idx_to_name[i] for i in idx_to_name
                                    if mdp.states[i]['frailty']==0 and mdp.states[i]['dose']==1],
        "Frail + Full at baseline":[idx_to_name[i] for i in idx_to_name
                                    if mdp.states[i]['frailty']==1 and mdp.states[i]['dose']==1],
        "Frail + Reduced at baseline":[idx_to_name[i] for i in idx_to_name
                                       if mdp.states[i]['frailty']==1 and mdp.states[i]['dose']==0],
    }
    return groups

def visualize_trajectory_by_baseline_groups(
    mdp, optimal_policy, trajectory_results,
    savefig='qol_trajectory_baseline_groups.png',
    policy_colors=None
):
    if policy_colors is None:
        policy_colors = {
            'optimal': 'red',
            'always_reduced': 'gold',
            'always_full': 'blue',
            'frailty_based': 'green',
            'clinical_test_based': 'orange',
        }

    groups = get_baseline_groups(mdp, trajectory_results)

    desired_titles = [
        "Frail + Full at baseline",
        "Frail + Reduced at baseline",
        "Fit + Full at baseline",
        "All patients (baseline)",
    ]
    titles = [t for t in desired_titles if t in groups] + \
             [t for t in groups.keys() if t not in desired_titles]

    fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=True)
    axes = axes.ravel()
    fig.suptitle('QOL Trajectories by Baseline Cohorts and different policies',
                 fontsize=16, fontweight='bold')

    y_min, y_max = 0.005, 0.13

    for i, (ax, title) in enumerate(zip(axes, titles)):
        states_in_group = groups[title]
        ax.set_title(f"{title}")

        for pname, color in policy_colors.items():
            m, s = _aggregate_group(trajectory_results, states_in_group, pname)
            if m is None:
                continue

            visit_idx = np.arange(len(m))   # quarters
            x_years = visit_idx * 0.25      # convert to years

            ax.plot(x_years, m, label=pname, linewidth=2,
                    color=color, alpha=0.9)
            ax.fill_between(x_years, m - s, m + s, alpha=0.18, color=color)

        ax.set_xlabel('Time (years)')
        ax.set_ylim(y_min, y_max)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8, ncol=2)

        # μόνο στα αριστερά plots (0 και 2) δείχνουμε y-label
        if i % 2 == 0:
            ax.set_ylabel('QOL or QALY/visit')
        else:
            ax.set_ylabel('')

    for j in range(len(titles), len(axes)):
        axes[j].axis('off')

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    if savefig:
        plt.savefig(savefig, dpi=300, bbox_inches='tight')


    


# ----- keep your existing visualize_trajectory_analysis(...) if you like -----
# (no changes needed)

# ===== modify add_trajectory_analysis to ALSO plot the 4 baseline groups =====
def add_trajectory_analysis(mdp, optimal_policy, n_simulations=500, n_steps=40):
    """
    Run end-to-end trajectory analysis and visuals.
    """
    print("\n" + "="*80)
    print("RUNNING TRAJECTORY ANALYSIS")
    print("="*80)

    # Build policy set
    print("Creating alternative policies...")
    alt_policies = create_alternative_policies(mdp, optimal_policy=optimal_policy)

    print(f"Simulating trajectories: {n_simulations} sims × {n_steps} visits × {mdp.n_states} start states × {len(alt_policies)} policies")
    results = analyze_trajectories_by_initial_state(
        mdp, optimal_policy, alt_policies, n_simulations=n_simulations, n_steps=n_steps
    )

    # Classic 2×3 dashboard (optional, keep if you want it)
    print("Creating general visuals...")
    visualize_trajectory_analysis(mdp, optimal_policy, results)

    # NEW: 4 baseline-cohort panels requested
    print("Creating baseline-cohort visuals (All / Fit+Full / Frail+Full / Frail+Reduced)...")
    visualize_trajectory_by_baseline_groups(mdp, optimal_policy, results)

    print_trajectory_summary(results, optimal_policy, alt_policies)

    print("\n" + "="*80)
    print("TRAJECTORY ANALYSIS COMPLETE!")
    print("="*80)
    return results

In [18]:
# Έχεις ήδη:
# mdp = SimpleQOLMDP(...); mdp.estimate_qol_rewards(...); mdp.estimate_transition_probabilities(...)
V_vi, pol_vi = mdp.value_iteration()

# Run trajectory analysis for 10 years (40 visits)
trajectory_results = add_trajectory_analysis(mdp, pol_vi, n_simulations=500, n_steps=40)


In [22]:
# --- helper: aggregate over an arbitrary list of state names for a given policy ---
def _aggregate_group(trajectory_results, state_names, policy_name):
    series = []
    T = None
    for st in state_names:
        if st in trajectory_results and policy_name in trajectory_results[st]:
            y = trajectory_results[st][policy_name]['mean_qol_over_time']
            series.append(np.array(y, dtype=float))
            T = len(y) if T is None else min(T, len(y))
    if not series:
        return None, None
    arr = np.vstack([s[:T] for s in series])
    return arr.mean(axis=0), arr.std(axis=0)


# --- helper: build baseline groups from mdp.states ---
def get_baseline_groups(mdp, trajectory_results):
    """
    Returns dict: {group_name: [state_names]} for the baseline cohorts:
      - Frail  (all frail, any dose)
      - Fit    (fit + full dose at baseline)
    """
    # map index->name that exists in trajectory_results
    idx_to_name = {
        i: mdp.states[i]['name']
        for i in range(mdp.n_states)
        if mdp.states[i]['name'] in trajectory_results
    }

    # all frail (dose 0 or 1)
    frail_states = [
        idx_to_name[i] for i in idx_to_name
        if mdp.states[i]['frailty'] == 1
    ]

    # fit + full dose at baseline
    fit_full_states = [
        idx_to_name[i] for i in idx_to_name
        if mdp.states[i]['frailty'] == 0 and mdp.states[i]['dose'] == 1
    ]

    groups = {
        "Frail": frail_states,
        "Fit": fit_full_states,
    }
    return groups


def visualize_trajectory_by_baseline_groups(
    mdp, optimal_policy, trajectory_results,
    savefig='qol_trajectory_baseline_groups.png',
    policy_colors=None
):
    # Χρωματισμοί για τις πολιτικές
    if policy_colors is None:
        policy_colors = {
            'optimal': 'red',
            'always_reduced': 'gold',
            'always_full': 'blue',
        }

    # «ανθρώπινα» labels
    policy_labels = {
        'optimal': 'optimal policy',
        'always_full': 'full dose',
        'always_reduced': 'reduced dose',
    }

    # groups: Frail & Fit (όπως ορίστηκαν στο get_baseline_groups)
    groups = get_baseline_groups(mdp, trajectory_results)
    titles = ["Frail", "Fit"]  # σειρά panels

    # Ποιες πολιτικές φαίνονται σε ποιο group
    allowed_policies_by_group = {
        "Frail": {"optimal", "always_full", "always_reduced"},
        "Fit":   {"optimal", "always_full",  "always_reduced"},
    }

    # 1x2 layout (Frail / Fit) με κοινό y-axis
    fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharey=True)
    fig.suptitle(
        'QALY trajectories for frail and fit patients under different policies',
        fontsize=24, fontweight='bold'
    )

    y_min, y_max = 0.005, 0.17

    for i, (ax, title) in enumerate(zip(axes, titles)):
        states_in_group = groups[title]
        ax.set_title(title, fontsize=18)

        allowed = allowed_policies_by_group[title]

        for pname, color in policy_colors.items():
            if pname not in allowed:
                continue

            m, s = _aggregate_group(trajectory_results, states_in_group, pname)
            if m is None:
                continue

            visit_idx = np.arange(len(m))      # 0..N-1 (quarters)
            x_months = visit_idx * 3.0         # 1 visit = 3 μήνες

            # ---- Κόβουμε τα ίδια τα δεδομένα στα 60 μήνες ----
            mask   = x_months <= 60
            x_plot = x_months[mask]
            m_plot = m[mask]
            s_plot = s[mask]

            label = policy_labels.get(pname, pname.replace('_', ' '))

            ax.plot(
                x_plot, m_plot,
                label=label,
                linewidth=3.0,
                color=color,
                alpha=0.9
            )
            ax.fill_between(
                x_plot, m_plot - s_plot, m_plot + s_plot,
                alpha=0.18, color=color
            )

        # ---- Χ άξονας: 0–60 μήνες, ticks ανά 3 μήνες ----
        ax.set_xlim(0, 60)
        ax.xaxis.set_major_locator(mticker.MultipleLocator(3.0))
        ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%d'))

        ax.set_xlabel('Time (months)', fontsize=16)
        if i == 0:
            ax.set_ylabel('QALYs per 3-month cycle', fontsize=16)

        ax.tick_params(axis='both', which='major', labelsize=14)
        ax.set_ylim(y_min, y_max)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=13, ncol=1, loc='upper left')

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    if savefig:
        plt.savefig(savefig, dpi=300, bbox_inches='tight')



# ===== add_trajectory_analysis: τρέχει simulation + φτιάχνει visuals =====
def add_trajectory_analysis(mdp, optimal_policy, n_simulations=500, n_steps=40):
    """
    Run end-to-end trajectory analysis and visuals.
    """
    print("\n" + "="*80)
    print("RUNNING TRAJECTORY ANALYSIS")
    print("="*80)

    # Build policy set
    print("Creating alternative policies...")
    alt_policies = create_alternative_policies(mdp, optimal_policy=optimal_policy)

    print(
        f"Simulating trajectories: {n_simulations} sims × {n_steps} visits × "
        f"{mdp.n_states} start states × {len(alt_policies)} policies"
    )
    results = analyze_trajectories_by_initial_state(
        mdp, optimal_policy, alt_policies,
        n_simulations=n_simulations, n_steps=n_steps
    )

    print("Creating general visuals...")
    visualize_trajectory_analysis(mdp, optimal_policy, results)

    print("Creating baseline-cohort visuals (Frail / Fit)...")
    visualize_trajectory_by_baseline_groups(mdp, optimal_policy, results)

    print_trajectory_summary(results, optimal_policy, alt_policies)

    print("\n" + "="*80)
    print("TRAJECTORY ANALYSIS COMPLETE!")
    print("="*80)
    return results


In [25]:
import matplotlib.ticker as mticker

# --- helper: aggregate over an arbitrary list of state names for a given policy ---
def _aggregate_group(trajectory_results, state_names, policy_name):
    series = []
    T = None
    for st in state_names:
        if st in trajectory_results and policy_name in trajectory_results[st]:
            y = trajectory_results[st][policy_name]['mean_qol_over_time']
            series.append(np.array(y, dtype=float))
            T = len(y) if T is None else min(T, len(y))
    if not series:
        return None, None
    arr = np.vstack([s[:T] for s in series])
    return arr.mean(axis=0), arr.std(axis=0)


# --- helper: build baseline groups from mdp.states ---
def get_baseline_groups(mdp, trajectory_results):
    """
    Returns dict: {group_name: [state_names]} for the baseline cohorts:
      - Frail  (all frail, any dose)
      - Fit    (fit + full dose at baseline)
    """
    # map index->name that exists in trajectory_results
    idx_to_name = {
        i: mdp.states[i]['name']
        for i in range(mdp.n_states)
        if mdp.states[i]['name'] in trajectory_results
    }

    # all frail (dose 0 or 1)
    frail_states = [
        idx_to_name[i] for i in idx_to_name
        if mdp.states[i]['frailty'] == 1
    ]

    # fit + full dose at baseline
    fit_full_states = [
        idx_to_name[i] for i in idx_to_name
        if mdp.states[i]['frailty'] == 0 and mdp.states[i]['dose'] == 1
    ]

    groups = {
        "Frail": frail_states,
        "Fit": fit_full_states,
    }
    return groups


def visualize_trajectory_by_baseline_groups(
    mdp, optimal_policy, trajectory_results,
    savefig='qol_trajectory_baseline_groups.png',
    policy_colors=None
):
    # Χρωματισμοί για τις πολιτικές
    if policy_colors is None:
        policy_colors = {
            'optimal': 'red',
            'always_reduced': 'gold',
            'always_full': 'blue',
        }

    # Labels για να φαίνονται «ανθρώπινα» στο legend
    policy_labels = {
        'optimal': 'optimal policy',
        'always_full': 'full dose',
        'always_reduced': 'reduced dose',
    }

    groups = get_baseline_groups(mdp, trajectory_results)

    desired_titles = ["Frail", "Fit"]
    titles = [t for t in desired_titles if t in groups] + \
             [t for t in groups.keys() if t not in desired_titles]

    # Επιτρεπόμενες πολιτικές ανά group
    allowed_policies_by_group = {
        "Frail": {"optimal", "always_full", "always_reduced"},
        "Fit": {"optimal", "always_full"},
    }

    fig, axes = plt.subplots(2, 2, figsize=(15, 11), sharex=True, sharey=True)
    axes = axes.ravel()
    fig.suptitle(
        'QOL trajectories for frail and fit patients under different policies',
        fontsize=24, fontweight='bold'
    )

    y_min, y_max = 0.005, 0.15

    for i, (ax, title) in enumerate(zip(axes, titles)):
        states_in_group = groups[title]
        ax.set_title(f"{title}", fontsize=18)

        allowed = allowed_policies_by_group.get(title, set(policy_colors.keys()))

        max_years = 0.0

        for pname, color in policy_colors.items():
            if pname not in allowed:
                continue

            m, s = _aggregate_group(trajectory_results, states_in_group, pname)
            if m is None:
                continue

            visit_idx = np.arange(len(m))   # quarters
            x_years = visit_idx * 0.25      # convert to years
            if len(x_years) > 0:
                max_years = max(max_years, x_years[-1])

            label = policy_labels.get(pname, pname.replace('_', ' '))

            ax.plot(
                x_years, m,
                label=label,
                linewidth=3.0,
                color=color,
                alpha=0.9
            )
            ax.fill_between(x_years, m - s, m + s, alpha=0.18, color=color)

        # -------- ΕΔΩ κλειδώνουμε τον άξονα σε 0–10 και βάζουμε 0,1,2,... ----------
        if max_years == 0:
            max_years = 5   # fallback αν για κάποιο λόγο δεν μπήκαν δεδομένα

        # αν θες ακριβώς 10 χρόνια:
        ax.set_xlim(0, 4)

        ax.xaxis.set_major_locator(mticker.MultipleLocator(1.0))
        ax.xaxis.set_minor_locator(mticker.NullLocator())
        ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%d'))

        # -------------------------------------------------------------------- #

        ax.set_xlabel('Time (years)', fontsize=16)
        if i % 2 == 0:
            ax.set_ylabel('QOL or QALY/visit', fontsize=16)
        else:
            ax.set_ylabel('')

        ax.tick_params(axis='both', which='major', labelsize=14)
        ax.set_ylim(y_min, y_max)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=13, ncol=2)

    # Σβήσε τα άδεια panels
    for j in range(len(titles), len(axes)):
        axes[j].axis('off')

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    if savefig:
        plt.savefig(savefig, dpi=300, bbox_inches='tight')



# ===== add_trajectory_analysis: τρέχει simulation + φτιάχνει visuals =====
def add_trajectory_analysis(mdp, optimal_policy, n_simulations=500, n_steps=40):
    """
    Run end-to-end trajectory analysis and visuals.
    """
    print("\n" + "="*80)
    print("RUNNING TRAJECTORY ANALYSIS")
    print("="*80)

    # Build policy set
    print("Creating alternative policies...")
    alt_policies = create_alternative_policies(mdp, optimal_policy=optimal_policy)

    print(
        f"Simulating trajectories: {n_simulations} sims × {n_steps} visits × "
        f"{mdp.n_states} start states × {len(alt_policies)} policies"
    )
    results = analyze_trajectories_by_initial_state(
        mdp, optimal_policy, alt_policies,
        n_simulations=n_simulations, n_steps=n_steps
    )

    print("Creating general visuals...")
    visualize_trajectory_analysis(mdp, optimal_policy, results)

    print("Creating baseline-cohort visuals (Frail / Fit)...")
    visualize_trajectory_by_baseline_groups(mdp, optimal_policy, results)

    print_trajectory_summary(results, optimal_policy, alt_policies)

    print("\n" + "="*80)
    print("TRAJECTORY ANALYSIS COMPLETE!")
    print("="*80)
    return results



In [26]:
# Run trajectory analysis for 10 years (40 visits)
trajectory_results = add_trajectory_analysis(mdp, pol_vi, n_simulations=500, n_steps=20)


In [36]:
# Run trajectory analysis for 10 years (40 visits)
trajectory_results = add_trajectory_analysis(mdp, pol_vi, n_simulations=500, n_steps=21)


In [37]:
import matplotlib.ticker as mticker

def visualize_cumulative_trajectory_by_baseline_groups(
    mdp, optimal_policy, trajectory_results,
    savefig='cumulative_qaly_baseline_groups.png',
    policy_colors=None
):
    """
    Όμοιο με το visualize_trajectory_by_baseline_groups,
    αλλά δείχνει cumulative QALYs αντί για QOL/QALY per visit.
    Χρόνος: 0–60 μήνες, ticks ανά 3 μήνες.
    """
    # Χρωματισμοί για τις πολιτικές – ίδιοι με πριν
    if policy_colors is None:
        policy_colors = {
            'optimal': 'red',
            'always_reduced': 'gold',
            'always_full': 'blue',
        }

    # «ανθρώπινα» labels
    policy_labels = {
        'optimal': 'optimal policy',
        'always_full': 'full dose',
        'always_reduced': 'reduced dose',
    }

    # groups: Frail & Fit (όπως ορίζονται στο get_baseline_groups)
    groups = get_baseline_groups(mdp, trajectory_results)
    titles = ["Frail", "Fit"]  # σειρά panels

    # Ποιες πολιτικές φαίνονται σε ποιο group
    allowed_policies_by_group = {
        "Frail": {"optimal", "always_full", "always_reduced"},
        "Fit":   {"optimal", "always_full"},
    }

    # 1x2 layout (Frail / Fit) με κοινό y-axis
    fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharey=True)
    fig.suptitle(
        'Cumulative QALYs for frail and fit patients under different policies',
        fontsize=24, fontweight='bold'
    )

    # 0.25 χρόνια ανά visit → 3 μήνες ανά visit
    VISIT_LEN_YEARS = 0.25

    for i, (ax, title) in enumerate(zip(axes, titles)):
        states_in_group = groups[title]
        ax.set_title(title, fontsize=18)

        allowed = allowed_policies_by_group[title]

        y_min, y_max = None, None

        for pname, color in policy_colors.items():
            if pname not in allowed:
                continue

            # Χρησιμοποιούμε το ΙΔΙΟ helper όπως στο QOL plot
            m, s = _aggregate_group(trajectory_results, states_in_group, pname)
            if m is None:
                continue

            visit_idx = np.arange(len(m))      # 0..N-1 (visits)
            x_months = visit_idx * 3.0         # 1 visit = 3 μήνες

            # --- cumulative QALYs: cumsum(QOL) * 0.25 χρόνια ανά visit ---
            cum_m = np.cumsum(m * VISIT_LEN_YEARS)
            cum_s = np.cumsum(s * VISIT_LEN_YEARS)  # approx για το shading

            # ---- Κόβουμε τα δεδομένα στα 60 μήνες ----
            mask    = x_months <= 60
            x_plot  = x_months[mask]
            m_plot  = cum_m[mask]
            s_plot  = cum_s[mask]

            if x_plot.size > 0:
                cur_min = (m_plot - s_plot).min()
                cur_max = (m_plot + s_plot).max()
                y_min = cur_min if y_min is None else min(y_min, cur_min)
                y_max = cur_max if y_max is None else max(y_max, cur_max)

            label = policy_labels.get(pname, pname.replace('_', ' '))

            ax.plot(
                x_plot, m_plot,
                label=label,
                linewidth=3.0,
                color=color,
                alpha=0.9
            )
            ax.fill_between(
                x_plot, m_plot - s_plot, m_plot + s_plot,
                alpha=0.18, color=color
            )

        # ---- Χ άξονας: 0–60 μήνες, ticks ανά 3 μήνες ----
        ax.set_xlim(0, 60)
        ax.xaxis.set_major_locator(mticker.MultipleLocator(3.0))
        ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%d'))

        # y-axis με λίγο περιθώριο
        if y_min is not None and y_max is not None:
            span = y_max - y_min if y_max > y_min else 1.0
            ax.set_ylim(y_min - 0.05*span, y_max + 0.05*span)

        ax.set_xlabel('Time (months)', fontsize=16)
        if i == 0:
            ax.set_ylabel('Cumulative QALYs', fontsize=16)

        ax.tick_params(axis='both', which='major', labelsize=14)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=13, ncol=1, loc='upper left')

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    if savefig:
        plt.savefig(savefig, dpi=300, bbox_inches='tight')



In [38]:
trajectory_results = add_trajectory_analysis(mdp, pol_vi, n_simulations=500, n_steps=21)

# cumulative plot:
visualize_cumulative_trajectory_by_baseline_groups(mdp, pol_vi, trajectory_results)


In [39]:
import matplotlib.ticker as mticker

def visualize_cumulative_trajectory_by_baseline_groups(
    mdp, optimal_policy, trajectory_results,
    savefig='cumulative_qaly_baseline_groups.png',
    policy_colors=None
):
    """
    Cumulative QALYs, υποθέτοντας ότι mean_qol_over_time είναι ήδη
    QALYs ανά τρίμηνο (per visit).
    """
    if policy_colors is None:
        policy_colors = {
            'optimal': 'red',
            'always_reduced': 'gold',
            'always_full': 'blue',
        }

    policy_labels = {
        'optimal': 'optimal policy',
        'always_full': 'full dose',
        'always_reduced': 'reduced dose',
    }

    groups = get_baseline_groups(mdp, trajectory_results)
    titles = ["Frail", "Fit"]

    allowed_policies_by_group = {
        "Frail": {"optimal", "always_full", "always_reduced"},
        "Fit":   {"optimal", "always_full"},
    }

    fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharey=True)
    fig.suptitle(
        'Cumulative QALYs for frail and fit patients under different dose policies',
        fontsize=24, fontweight='bold'
    )

    for i, (ax, title) in enumerate(zip(axes, titles)):
        states_in_group = groups[title]
        ax.set_title(title, fontsize=18)

        allowed = allowed_policies_by_group[title]
        y_min, y_max = None, None

        for pname, color in policy_colors.items():
            if pname not in allowed:
                continue

            # ίδιο helper: παίρνουμε το per-visit "qol"
            m, s = _aggregate_group(trajectory_results, states_in_group, pname)
            if m is None:
                continue

            visit_idx = np.arange(len(m))      # 0..N-1 visits
            x_months = visit_idx * 3.0         # 1 visit = 3 μήνες

            # *** ΕΔΩ η διαφορά: cumulative = cumsum(m), ΟΧΙ m*0.25 ***
            cum_m = np.cumsum(m)
            cum_s = np.cumsum(s)   # approx για το shading

            # κόβουμε στα 60 μήνες
            mask    = x_months <= 60
            x_plot  = x_months[mask]
            m_plot  = cum_m[mask]
            s_plot  = cum_s[mask]

            if x_plot.size > 0:
                cur_min = (m_plot - s_plot).min()
                cur_max = (m_plot + s_plot).max()
                y_min = cur_min if y_min is None else min(y_min, cur_min)
                y_max = cur_max if y_max is None else max(y_max, cur_max)

            label = policy_labels.get(pname, pname.replace('_', ' '))

            ax.plot(x_plot, m_plot,
                    label=label, linewidth=3.0,
                    color=color, alpha=0.9)
            ax.fill_between(
                x_plot, m_plot - s_plot, m_plot + s_plot,
                alpha=0.18, color=color
            )

        # x-axis: 0–60 μήνες, ανά 3 μήνες
        ax.set_xlim(0, 60)
        ax.xaxis.set_major_locator(mticker.MultipleLocator(3.0))
        ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%d'))

        if y_min is not None and y_max is not None:
            span = y_max - y_min if y_max > y_min else 1.0
            ax.set_ylim(y_min - 0.05*span, y_max + 0.05*span)

        ax.set_xlabel('Time (months)', fontsize=16)
        if i == 0:
            ax.set_ylabel('Cumulative QALYs', fontsize=16)

        ax.tick_params(axis='both', which='major', labelsize=14)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=13, ncol=1, loc='upper left')

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    if savefig:
        plt.savefig(savefig, dpi=300, bbox_inches='tight')


In [40]:
import matplotlib.ticker as mticker

def visualize_cumulative_trajectory_by_baseline_groups(
    mdp, optimal_policy, trajectory_results,
    savefig='cumulative_qaly_baseline_groups.png',
    policy_colors=None
):
    """
    Cumulative QALYs, υποθέτοντας ότι mean_qol_over_time είναι ήδη
    QALYs ανά τρίμηνο (per visit).
    """
    if policy_colors is None:
        policy_colors = {
            'optimal': 'red',
            'always_reduced': 'gold',
            'always_full': 'blue',
        }

    policy_labels = {
        'optimal': 'optimal policy',
        'always_full': 'full dose',
        'always_reduced': 'reduced dose',
    }

    groups = get_baseline_groups(mdp, trajectory_results)
    titles = ["Frail", "Fit"]

    allowed_policies_by_group = {
        "Frail": {"optimal", "always_full", "always_reduced"},
        "Fit":   {"optimal", "always_full"},
    }

    fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharey=True)
    fig.suptitle(
        'Cumulative QALYs for frail and fit patients under different dose policies',
        fontsize=24, fontweight='bold'
    )

    for i, (ax, title) in enumerate(zip(axes, titles)):
        states_in_group = groups[title]
        ax.set_title(title, fontsize=18)

        allowed = allowed_policies_by_group[title]
        y_min, y_max = None, None

        for pname, color in policy_colors.items():
            if pname not in allowed:
                continue

            # ίδιο helper: παίρνουμε το per-visit "qol"
            m, s = _aggregate_group(trajectory_results, states_in_group, pname)
            if m is None:
                continue

            visit_idx = np.arange(len(m))      # 0..N-1 visits
            x_months = visit_idx * 3.0         # 1 visit = 3 μήνες

            # *** ΕΔΩ η διαφορά: cumulative = cumsum(m), ΟΧΙ m*0.25 ***
            cum_m = np.cumsum(m)
            cum_s = np.cumsum(s)   # approx για το shading

            # κόβουμε στα 60 μήνες
            mask    = x_months <= 60
            x_plot  = x_months[mask]
            m_plot  = cum_m[mask]
            s_plot  = cum_s[mask]

            if x_plot.size > 0:
                cur_min = (m_plot - s_plot).min()
                cur_max = (m_plot + s_plot).max()
                y_min = cur_min if y_min is None else min(y_min, cur_min)
                y_max = cur_max if y_max is None else max(y_max, cur_max)

            label = policy_labels.get(pname, pname.replace('_', ' '))

            ax.plot(x_plot, m_plot,
                    label=label, linewidth=3.0,
                    color=color, alpha=0.9)
            ax.fill_between(
                x_plot, m_plot - s_plot, m_plot + s_plot,
                alpha=0.18, color=color
            )

        # x-axis: 0–60 μήνες, ανά 3 μήνες
        ax.set_xlim(0, 60)
        ax.xaxis.set_major_locator(mticker.MultipleLocator(3.0))
        ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%d'))

        if y_min is not None and y_max is not None:
            span = y_max - y_min if y_max > y_min else 1.0
            ax.set_ylim(y_min - 0.05*span, y_max + 0.05*span)

        ax.set_xlabel('Time (months)', fontsize=16)
        if i == 0:
            ax.set_ylabel('Cumulative QALYs', fontsize=16)

        ax.tick_params(axis='both', which='major', labelsize=14)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=13, ncol=1, loc='upper left')

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    if savefig:
        plt.savefig(savefig, dpi=300, bbox_inches='tight')


In [41]:
trajectory_results = add_trajectory_analysis(mdp, pol_vi, n_simulations=500, n_steps=21)
visualize_cumulative_trajectory_by_baseline_groups(mdp, pol_vi, trajectory_results)


In [43]:
def _aggregate_group(trajectory_results, state_names, policy_name):
    """
    Patient-level aggregation:
    - mean(t): mean των mean_qol_over_time(t) across states
    - std(t): std πάνω σε ΟΛΕΣ τις προσομοιώσεις των states (law of total variance)
    """
    mus = []
    sigs = []
    T = None

    for st in state_names:
        if st not in trajectory_results:
            continue
        if policy_name not in trajectory_results[st]:
            continue

        res = trajectory_results[st][policy_name]
        mu  = np.array(res['mean_qol_over_time'], dtype=float)
        sig = np.array(res['std_qol_over_time'],  dtype=float)

        if T is None:
            T = len(mu)
        else:
            T = min(T, len(mu))

        mus.append(mu[:T])
        sigs.append(sig[:T])

    if not mus:
        return None, None

    mus  = np.vstack(mus)   # shape: (S, T)
    sigs = np.vstack(sigs)  # shape: (S, T)

    # mean QALYs per cycle στο cohort
    mean = mus.mean(axis=0)   # E_s[μ_s(t)]

    # Var_total(t) = E_s[σ_s^2 + μ_s^2] - (E_s[μ_s])^2
    mean_mu2_plus_sig2 = (sigs**2 + mus**2).mean(axis=0)
    var  = mean_mu2_plus_sig2 - mean**2
    var  = np.clip(var, 0, None)
    std  = np.sqrt(var)

    return mean, std


In [44]:
# Run trajectory analysis for 10 years (40 visits)
trajectory_results = add_trajectory_analysis(mdp, pol_vi, n_simulations=500, n_steps=21)

In [45]:
def print_optimal_policy(mdp, policy):
    print("\n" + "="*60)
    print("OPTIMAL POLICY (per MDP state)")
    print("="*60)
    print(f"{'State ID':<8} {'State name':<18} {'frail':<5} {'test':<5} {'current_dose':<12} {'action*':<10}")
    print("-"*60)

    for s in range(mdp.n_states):
        st = mdp.states[s]
        a = int(policy[s])
        action_name = mdp.actions[a]          # 'reduced_dose' ή 'full_dose'
        cur_dose    = 'full' if st['dose']==1 else 'reduced'
        test_label  = 'pos' if st['clinical_test']==1 else 'neg'
        frail_label = 'frail' if st['frailty']==1 else 'fit'

        print(f"{s:<8} {st['name']:<18} {frail_label:<5} {test_label:<5} {cur_dose:<12} {action_name:<10}")


In [46]:
V_opt, optimal_policy = mdp.value_iteration()
print_optimal_policy(mdp, optimal_policy)


In [57]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker


def visualize_trajectory_by_baseline_groups(
    mdp,
    optimal_policy,
    trajectory_results,
    savefig="Figure_5_corrected.png",
    policy_colors=None,
    max_months=60
):
    """
    Visualize QALY trajectories for frail and fit baseline groups.

    Parameters
    ----------
    max_months : int or None
        Maximum displayed follow-up time in months.
        Use 60 to reproduce the five-year figure.
        Use None to display the full simulated trajectory.
    """

    if policy_colors is None:
        policy_colors = {
            "optimal": "red",
            "always_reduced": "gold",
            "always_full": "blue",
        }

    policy_labels = {
        "optimal": "optimal policy",
        "always_reduced": "reduced dose",
        "always_full": "full dose",
    }

    groups = get_baseline_groups(mdp, trajectory_results)

    # Κρατάμε μόνο τα δύο panels που χρειαζόμαστε
    titles = [title for title in ("Frail", "Fit") if title in groups]

    if not titles:
        raise ValueError(
            "No 'Frail' or 'Fit' baseline groups were found."
        )

    # Και οι τρεις πολιτικές εμφανίζονται και στις δύο ομάδες
    allowed_policies_by_group = {
        "Frail": {"optimal", "always_full", "always_reduced"},
        "Fit": {"optimal", "always_full", "always_reduced"},
    }

    fig, axes = plt.subplots(
        1,
        len(titles),
        figsize=(16, 6),
        sharex=True,
        sharey=True,
        squeeze=False
    )

    axes = axes.ravel()

    y_min, y_max = 0.005, 0.17

    for ax, title in zip(axes, titles):
        states_in_group = groups[title]
        allowed_policies = allowed_policies_by_group[title]

        group_max_months = 0

        for policy_name, color in policy_colors.items():
            if policy_name not in allowed_policies:
                continue

            mean_values, spread_values = _aggregate_group(
                trajectory_results,
                states_in_group,
                policy_name
            )

            if mean_values is None:
                continue

            mean_values = np.asarray(mean_values)
            spread_values = np.asarray(spread_values)

            # Κάθε visit αντιστοιχεί σε τρίμηνο
            visit_idx = np.arange(len(mean_values))
            x_months = visit_idx * 3

            if len(x_months) > 0:
                group_max_months = max(
                    group_max_months,
                    float(x_months[-1])
                )

            label = policy_labels.get(
                policy_name,
                policy_name.replace("_", " ")
            )

            ax.plot(
                x_months,
                mean_values,
                label=label,
                linewidth=3.0,
                color=color,
                alpha=0.9
            )

            ax.fill_between(
                x_months,
                mean_values - spread_values,
                mean_values + spread_values,
                color=color,
                alpha=0.18
            )

        ax.set_title(title, fontsize=20)

        # Axis labels και στα δύο panels
        ax.set_xlabel("Time (months)", fontsize=17)
        ax.set_ylabel("QALYs per 3-month cycle", fontsize=17)

        # Εμφάνιση y-axis tick labels και στο δεξί panel
        ax.tick_params(
            axis="both",
            which="major",
            labelsize=13,
            labelleft=True
        )

        if max_months is not None:
            ax.set_xlim(0, max_months)

            # Για 0–60 μήνες: ticks ανά 3 μήνες, όπως στο αρχικό figure
            if max_months <= 60:
                ax.xaxis.set_major_locator(
                    mticker.MultipleLocator(3)
                )
            else:
                ax.xaxis.set_major_locator(
                    mticker.MultipleLocator(12)
                )
        else:
            displayed_max = group_max_months or 60
            ax.set_xlim(0, displayed_max)
            ax.xaxis.set_major_locator(
                mticker.MultipleLocator(12)
            )

        ax.xaxis.set_major_formatter(
            mticker.FormatStrFormatter("%d")
        )

        ax.set_ylim(y_min, y_max)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=13, loc="best")

    fig.suptitle(
        "QALY trajectories for frail and fit patients under different policies",
        fontsize=22,
        fontweight="bold"
    )

    plt.tight_layout(rect=[0, 0, 1, 0.94])

    if savefig:
        plt.savefig(
            savefig,
            dpi=300,
            bbox_inches="tight"
        )

    plt.show()

    return fig, axes

In [59]:
# Run trajectory analysis for 10 years (40 visits)
trajectory_results = add_trajectory_analysis(mdp, pol_vi, n_simulations=500, n_steps=21)

In [63]:
savefig="Figure_5_corrected.png"